In [ ]:
# INQUIRE toxicity-driven prioritization - Part 2: Modeling
"""
Standalone. Reads the files saved by Part 1 (most recent run, or `PT1_DATE`), 
then models one tox endpoint at a time (set `ENDPOINT_SHORT` in the endpoint selection cell) 
and runs the cross-endpoint summaries once all endpoints are done.
"""


In [ ]:
# Reset env to erase previous vars, dicts, dfs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# Imports & setup
import warnings
from datetime import datetime
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import random
from tqdm.auto import tqdm

from scipy.stats import skew, rankdata # rankdata for SHAP direction

import shap
import xgboost as xgb
from xgboost import XGBRegressor
from joblib import Parallel, delayed

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score, roc_auc_score # roc_auc_score for yes/no endpoints
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import KFold, StratifiedKFold # StratifiedKFold
from sklearn.preprocessing import MinMaxScaler

# Hide Elastic Net convergence warnings (they flood the log in CV/permutation loops)
warnings.filterwarnings("ignore", category=ConvergenceWarning)


# Logging setup (equivalent to logger + appender_tee)
# log_file = f"inquire_tip_pt2_modeling_{datetime.now().strftime('%y%m%d')}.log"
log_file = f"inquire_tip_pt2_modeling_{datetime.now().strftime('%y%m%d_%H%M%S')}.log" # date + time: a new run never overwrites the log of an earlier run

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler(log_file, mode="w"),
        logging.StreamHandler()
    ],
    force=True # re-applies the setup on re-runs (without it, the log file is not re-created after %reset)
)

# Helper functions
def majorsep():
    logging.info("=" * 100)

def minorsep():
    logging.info("-" * 100)

# Validation helpers
def check_file(path_obj, label):
    if not path_obj.exists():
        raise FileNotFoundError(f" - {label} missing: {path_obj}")
    logging.info(f" - Found file: {label}")

def safe_read_csv(path_obj, label, **kwargs):
    if "low_memory" not in kwargs:
        kwargs["low_memory"] = False
    check_file(path_obj, label)
    try:
        df = pd.read_csv(path_obj, **kwargs)
        logging.info(f" - Loaded {label} | rows={len(df)} cols={len(df.columns)}")
        return df
    except Exception as e:
        raise RuntimeError(f" - Failed loading {label}: {e}")

# Modeling helpers
def plan_endpoint(y, censor_frac=0.35, min_class_n=15, skew_log=1.0):
    """Suggests how to model one tox endpoint.
    - Many samples share the lowest value (e.g. <LOQ replaced by LOQ/sqrt(2), or 0 = no effect)
      → 'classification': active (above that value) vs. inactive.
    - Too few active OR inactive samples for that → 'skip'.
    - Otherwise 'regression', on log10 scale if strongly right-skewed."""
    y = pd.to_numeric(pd.Series(y), errors="coerce").dropna()
    if len(y) == 0:
        return dict(task="skip", transform="none", n=0, n_unique=0, mode_value=np.nan, n_mode=0,
                    frac_mode=np.nan, mode_is_min=False, n_active=0, skewness=np.nan, reason="no values")
    counts = y.value_counts()
    mode_val, n_mode = counts.index[0], int(counts.iloc[0])
    frac_mode = n_mode / len(y)
    mode_is_min = bool(np.isclose(mode_val, y.min()))
    n_active = int((~np.isclose(y, y.min())).sum())
    sk = float(skew(y)) if y.nunique() > 2 else np.nan

    if frac_mode >= censor_frac and mode_is_min:
        n_minor = min(n_active, len(y) - n_active)
        if n_minor >= min_class_n:
            task, reason = "classification", f"{frac_mode:.0%} of samples at lowest value"
        else:
            task, reason = "skip", f"only {n_minor} samples in the smaller class (< {min_class_n})"
        transform = "binary"
    else:
        task, reason = "regression", "continuous"
        transform = "log10" if (sk > skew_log and (y >= 0).all()) else "none"

    return dict(task=task, transform=transform, n=len(y), n_unique=int(y.nunique()), mode_value=mode_val,
                n_mode=n_mode, frac_mode=frac_mode, mode_is_min=mode_is_min, n_active=n_active,
                skewness=sk, reason=reason)

def r_scale(df):
    """Z-score like R's scale(): (x - mean) / sample SD (n - 1). Drops columns with SD = 0 or undefined."""
    sd = df.std(ddof=1)
    keep = sd.index[(sd > 0) & sd.notna()]
    return (df[keep] - df[keep].mean()) / sd[keep], sd.index.difference(keep)

def make_cv(task, n_splits, seed):
    """Shuffled K-fold; stratified for classification so every fold has both classes."""
    if task == "classification":
        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return KFold(n_splits=n_splits, shuffle=True, random_state=seed)

def score_metric(task, y_true, y_pred):
    """Q² (R² of out-of-fold predictions) for regression; ROC AUC for classification."""
    if task == "classification":
        return roc_auc_score(y_true, y_pred)
    return r2_score(y_true, y_pred)

def repeated_cv_metric(model, X, y, task, n_splits, n_repeats, seed, n_jobs=-1):
    """One pooled out-of-fold metric per repeat (each sample predicted once per repeat).
    Much more stable than averaging per-fold R² on ~30 test samples."""
    scores = []
    for r in tqdm(range(n_repeats), desc="Repeated CV", leave=False):
        pred = cross_val_predict(clone(model), X, y, cv=make_cv(task, n_splits, seed + r), n_jobs=n_jobs)
        scores.append(score_metric(task, y, pred))
    return np.array(scores)

def _perm_one(model, X, y, task, n_splits, seed, perm_seed):
    y_perm = np.random.default_rng(perm_seed).permutation(y)
    pred = cross_val_predict(clone(model), X, y_perm, cv=make_cv(task, n_splits, seed))
    return score_metric(task, y_perm, pred)

# Progress bars for parallel jobs (permutation tests, stability selection)
def _indexed_call(func, i, args):
    return i, func(*args)

def run_parallel(func, args_list, desc, n_jobs=-1):
    """Runs func(*args) for every args in args_list in parallel, with a progress bar
    (elapsed time + estimated time left). Results are returned in the original order."""
    results = [None] * len(args_list)
    tasks = [delayed(_indexed_call)(func, i, a) for i, a in enumerate(args_list)]
    bar = tqdm(total=len(tasks), desc=desc)
    try:
        out = Parallel(n_jobs=n_jobs, return_as="generator_unordered")(tasks) # joblib >= 1.4
    except (TypeError, ValueError):
        try:
            out = Parallel(n_jobs=n_jobs, return_as="generator")(tasks) # joblib 1.3
        except (TypeError, ValueError):
            logging.warning(f" - Old joblib version: progress bar only updates at the end (update joblib for live progress)")
            out = Parallel(n_jobs=n_jobs)(tasks)
    for i, res in out:
        results[i] = res
        bar.update(1)
    bar.close()
    return results

def permutation_test_metric(model, X, y, task, n_splits, n_perm, seed, n_jobs=-1, desc="Permutation test"):
    """Observed pooled CV metric vs. the same metric after shuffling y.
    p = (n permuted scores >= observed + 1) / (n_perm + 1); smallest possible p = 1 / (n_perm + 1)."""
    pred = cross_val_predict(clone(model), X, y, cv=make_cv(task, n_splits, seed), n_jobs=n_jobs)
    observed = score_metric(task, y, pred)
    perm_scores = np.array(run_parallel(
        _perm_one,
        [(model, X, y, task, n_splits, seed, seed + 1000 + i) for i in range(n_perm)],
        desc=f"{desc} ({n_perm} permutations)",
        n_jobs=n_jobs
    ))
    pvalue = (np.sum(perm_scores >= observed) + 1) / (n_perm + 1)
    return observed, perm_scores, pvalue

# Metadata from the full classified metadata (feat_meta_class) if loaded;
# feat_meta_dust_masked only has features kept by the detection filter, so rarer features got no metadata
def add_meta(df):
    """Adds feature metadata to a table with an ML_feature_id column."""
    meta = None
    for meta_name in ["feat_meta_class", "feat_meta_dust_masked"]:
        m = globals().get(meta_name)
        if m is not None and "ML_feature_id" in m.columns:
            meta = m
            break
    if meta is None or "ML_feature_id" not in df.columns:
        return df
    meta = meta.copy()
    meta["ML_feature_id"] = meta["ML_feature_id"].astype(str).str.strip()
    meta = meta.drop_duplicates("ML_feature_id")
    meta = meta[["ML_feature_id"] + [c for c in meta.columns if c not in df.columns]]
    out = df.copy()
    out["ML_feature_id"] = out["ML_feature_id"].astype(str).str.strip()
    return out.merge(meta, on="ML_feature_id", how="left")

majorsep()
logging.info(f"joblib version: {joblib.__version__ if 'joblib' in globals() else __import__('joblib').__version__}")
minorsep()

def make_tuned_enet(task, l1_ratio, alphas, seed, inner_splits=3):
    """Elastic Net that re-tunes alpha on whatever training data it is given (nested CV).
    Used for CV and permutation tests: tuning alpha once on all samples and re-using it would let the real y
    leak into the CV metric and make the permutation test too optimistic (shuffled y never gets a tuned alpha)."""
    return ElasticNetCV(l1_ratio=l1_ratio, alphas=alphas, cv=make_cv(task, inner_splits, seed), max_iter=10000)

class ENetTopKSelector(BaseEstimator, TransformerMixin):
    """Feature selection that is re-fitted INSIDE every CV training fold (no information from test samples).
    Keeps features with a non-zero Elastic Net coefficient (largest |coef| first, max k).
    If fewer than min_features are non-zero, tops up with the features most correlated with y."""
    def __init__(self, enet=None, k=200, min_features=10):
        self.enet = enet # an (un-fitted) Elastic Net model, e.g. make_tuned_enet(...)
        self.k = k
        self.min_features = min_features

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        enet = clone(self.enet) if self.enet is not None else ElasticNet(max_iter=10000)
        coef = enet.fit(X, y).coef_
        Xc, yc = X - X.mean(axis=0), y - y.mean()
        denom = np.sqrt((Xc ** 2).sum(axis=0) * (yc ** 2).sum())
        corr = np.divide(np.abs(Xc.T @ yc), denom, out=np.zeros(X.shape[1]), where=denom > 0)
        abs_coef = np.abs(coef)
        score = np.where(abs_coef > 0, 2 + abs_coef / max(abs_coef.max(), 1e-12), corr) # non-zero coef always ranks first
        n_keep = int(np.clip((abs_coef > 0).sum(), min(self.min_features, X.shape[1]), min(self.k, X.shape[1])))
        self.support_idx_ = np.sort(np.argsort(score)[::-1][:n_keep])
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.support_idx_]

def save_with_meta(df, name):
    """Saves <name>.csv and <name>_w_metadata.csv to END_OUT_DIR."""
    df.to_csv(END_OUT_DIR / f"{name}.csv", index=False)
    add_meta(df).to_csv(END_OUT_DIR / f"{name}_w_metadata.csv", index=False)


# Seed
GLOBAL_SEED = 1
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

logging.info(f"Imports & logging setup done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Load preprocessed data (outputs of INQUIRE_TIP_Pt1_Preprocessing)
# - Uses the files of one Pt1 run (same yymmdd prefix): the most recent complete run, or PT1_DATE
# - X_tox_vs_dust_prep: detection-filtered + log10 dust matrix (scaled per endpoint at endpoint selection)
# - X_tox_vs_dust: raw numeric dust data, all features (univariate screen)
# - Y_tox_vs_dust: tox endpoints, unscaled (transformed per endpoint at endpoint selection)
majorsep()
logging.info(f"Loading preprocessed data (Pt1 outputs) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Folders (same as in Pt1)
DATA_DIR = Path("../data") # working folder (data/ next to notebooks/); or the full path, e.g. Path(r"D:/projects/INQUIRE_TIP/data")
OUT_DIR = Path(DATA_DIR / "inquire_tip_outputs")

PT1_DATE = None # yymmdd prefix of the Pt1 files to use, e.g. "261018"; None = most recent complete Pt1 run

PT1_FILES = {
    "settings": "preprocessing_settings.csv",
    "Y": "tox_endpoints_unscaled.csv",
    "X_prep": "dust_detfilt_log10.csv",
    "X_raw": "dust_raw_numeric.csv",
    "meta_class": "feature_metadata_class_clean.csv",
    "meta_masked": "feature_metadata_dust_masked.csv",
}

if PT1_DATE is None:
    runs = sorted(OUT_DIR.glob("*_" + PT1_FILES["settings"]))
    if not runs:
        raise FileNotFoundError(f"No Pt1 outputs found in {OUT_DIR}: run INQUIRE_TIP_Pt1_Preprocessing first")
    PT1_DATE = runs[-1].name.split("_")[0]

def pt1_file(key):
    return OUT_DIR / f"{PT1_DATE}_{PT1_FILES[key]}"

missing_pt1 = [pt1_file(k).name for k in PT1_FILES if not pt1_file(k).exists()]
if missing_pt1:
    raise FileNotFoundError(f"Pt1 files missing for run {PT1_DATE}: {missing_pt1}. Rerun Pt1 (incl. the save cell).")
logging.info(f" - Pt1 run used: {PT1_DATE}")
minorsep()

# Pt1 settings
pt1_settings = pd.read_csv(pt1_file("settings")).set_index("setting")["value"]
NEG_HANDLING = str(pt1_settings["NEG_HANDLING"])
NAN_TO_ZERO = str(pt1_settings["NAN_TO_ZERO"]) == "True"
MIN_DETECT_FRAC = float(pt1_settings["MIN_DETECT_FRAC"])
MIN_UNIQUE_VALUES = int(float(pt1_settings["MIN_UNIQUE_VALUES"]))
logging.info(f"Pt1 settings:")
for k, v in pt1_settings.items():
    logging.info(f" - {k}: {v}")
minorsep()

# Data
Y_tox_vs_dust = safe_read_csv(pt1_file("Y"), "Tox endpoints (unscaled)", index_col=0)
X_tox_vs_dust_prep = safe_read_csv(pt1_file("X_prep"), "Dust matrix (detection-filtered, log10)", index_col=0)
X_tox_vs_dust = safe_read_csv(pt1_file("X_raw"), "Dust matrix (raw numeric, all features)", index_col=0)
feat_meta_class = safe_read_csv(pt1_file("meta_class"), "Classified feature metadata", dtype={"ML_feature_id": str})
feat_meta_dust_masked = safe_read_csv(pt1_file("meta_masked"), "Feature metadata (modeled features)", dtype={"ML_feature_id": str})
minorsep()

for df_ in (Y_tox_vs_dust, X_tox_vs_dust_prep, X_tox_vs_dust):
    df_.index = df_.index.astype(str).str.strip()
    df_.columns = [str(c).strip() for c in df_.columns]
for df_ in (feat_meta_class, feat_meta_dust_masked):
    df_["ML_feature_id"] = df_["ML_feature_id"].astype(str).str.strip()
Y_tox_vs_dust = Y_tox_vs_dust.apply(pd.to_numeric, errors="coerce")
X_tox_vs_dust = X_tox_vs_dust.apply(pd.to_numeric, errors="coerce")

# Checks: same samples in the same order, no NaNs in the modeling matrix
if not Y_tox_vs_dust.index.equals(X_tox_vs_dust_prep.index):
    common_smpl = X_tox_vs_dust_prep.index.intersection(Y_tox_vs_dust.index)
    logging.warning(f" - Sample IDs/order differ between X and Y; using the {len(common_smpl)} samples in both (X order)")
    X_tox_vs_dust_prep, Y_tox_vs_dust = X_tox_vs_dust_prep.loc[common_smpl], Y_tox_vs_dust.loc[common_smpl]
X_tox_vs_dust = X_tox_vs_dust.reindex(X_tox_vs_dust_prep.index)
n_nan_x = int(X_tox_vs_dust_prep.isna().sum().sum())
if n_nan_x > 0:
    raise ValueError(f"{n_nan_x} NaNs in the modeling matrix: check NAN_TO_ZERO in Pt1")

# Scaled matrix over all samples (R scale()); per-endpoint scaling happens at endpoint selection
X_tox_vs_dust_scaled, _ = r_scale(X_tox_vs_dust_prep)

logging.info(f" - Y_tox_vs_dust [samples, endpoints]: {Y_tox_vs_dust.shape}")
logging.info(f" - X_tox_vs_dust_prep [samples, features]: {X_tox_vs_dust_prep.shape}")
logging.info(f" - X_tox_vs_dust (raw) [samples, features]: {X_tox_vs_dust.shape}")
logging.info(f" - Metadata: {len(feat_meta_class):,} features (classified), {len(feat_meta_dust_masked):,} (modeled)")
logging.info(f" - Samples aligned: X and Y have the same samples in the same order")

minorsep()
logging.info(f"Done loading preprocessed data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Endpoint selection
# - Pick the endpoint with ENDPOINT_SHORT only (full names in ENDPOINTS)
# - All run settings in one place; FAST_MODE = quick check (~3-4x faster, less precise)
# - Modeling plan per endpoint: 
#   - regression on log10 (skewed)
#   - classification (many samples at the lowest value = <LOQ substitute or "no effect")
#   - skip endpoint (too few active samples)
# - X is re-scaled on this endpoint's samples and features constant within them are dropped
majorsep()
logging.info(f"Picking endpoint to model | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


# Endpoint to model
ENDPOINT_SHORT = "AhR"   # replace with endpoint to model short name


ENDPOINTS = {
    "cell_viability": "reduced cell viability/mg BEAS-2B (effect cut-off = 20%)",
    "IL6_0.1": "relative IL-6 expression (BEAS-2B 0.1 mg/ml)",
    "IL6_1.0": "relative IL-6 expression (BEAS-2B 1 mg/ml)",
    "IL8_0.1": "relative IL-8 expression (BEAS-2B 0.1 mg/ml)",
    "IL8_1.0": "relative IL-8 expression (BEAS-2B 1 mg/ml)",
    "andro": "andro (ng/g)",
    "antiandro": "antiandro (µg/g)",
    "estro": "estro (pg/g)",
    "antiestro": "antiestro (ng/g)",
    "AhR": "AhR (ng/g)",
    "TR": "TR agonism (ng/g)",
}

# Modeling plan
TASK_OVERRIDE = None # None = automatic (recommended), or "regression" / "classification"
FORCE_RUN = True # True = also model an endpoint the plan marks as "skip"

# Run settings
FAST_MODE = False # True = quick check (fewer repeats/permutations/stability runs)
N_SPLITS = 5 # CV folds
N_REPEATS = 3 if FAST_MODE else 10 # CV repeats (5 folds x 10 repeats)
N_PERM = 20 if FAST_MODE else 100 # permutations for p-values (min p = 1 / (N_PERM + 1))
N_STABILITY_RUNS = 50 if FAST_MODE else 100
SUBSAMPLE_FRACTION = 0.75 # fraction of samples per stability run
STABILITY_THRESHOLD = 0.6 # "stable" features
HIGH_CONF_THRESHOLD = 0.8 # "high-confidence" features
XGB_MAX_FEATURES = 200 # max features passed to XGBoost (~160 samples cannot support thousands)
TOP_N_FSACI = 120
PRIORITIZE_POSITIVE_ONLY = True # FSACI ranks only compounds linked to higher endpoint values (more toxic)

# Checks
if ENDPOINT_SHORT not in ENDPOINTS:
    raise ValueError(f"Unknown ENDPOINT_SHORT '{ENDPOINT_SHORT}'. Options: {list(ENDPOINTS)}")
ENDPOINT = ENDPOINTS[ENDPOINT_SHORT]
if ENDPOINT not in Y_tox_vs_dust.columns:
    raise ValueError(f"Endpoint column '{ENDPOINT}' not found in Y_tox_vs_dust")

# Store current endpoint
Y_endpoint = Y_tox_vs_dust[[ENDPOINT]].copy()

# Drop samples with NaN for this endpoint
n_samples_before = len(Y_endpoint)
Y_endpoint = Y_endpoint.dropna()
logging.info(f" - Samples dropped (NaN in {ENDPOINT_SHORT}): {n_samples_before - len(Y_endpoint)} out of {n_samples_before}")

# Keep the same samples, in the same order, in the dust data
common_samples = Y_endpoint.index.intersection(X_tox_vs_dust_scaled.index)
Y_endpoint = Y_endpoint.loc[common_samples]
logging.info(f" - Samples in both X & Y used for modeling: {len(common_samples)}")
minorsep()

# Modeling plan
ENDPOINT_PLAN = plan_endpoint(Y_endpoint[ENDPOINT])
TASK = TASK_OVERRIDE if TASK_OVERRIDE else ENDPOINT_PLAN["task"]
logging.info(f" - Suggested task: {ENDPOINT_PLAN['task']} ({ENDPOINT_PLAN['reason']})")

if TASK == "skip":
    if not FORCE_RUN:
        raise RuntimeError(
            f"Endpoint '{ENDPOINT_SHORT}' skipped: {ENDPOINT_PLAN['reason']}. "
            f"Too little variation to link to compounds. Set FORCE_RUN = True to model it anyway (results unreliable)."
        )
    TASK = "classification"
    logging.warning(f" !!! FORCE_RUN: modeling a 'skip' endpoint as classification; results unreliable")

y_raw = Y_endpoint[ENDPOINT].astype(float)
if TASK == "classification":
    if ENDPOINT_PLAN["mode_is_min"] and ENDPOINT_PLAN["frac_mode"] >= 0.10:
        Y_CUTOFF = float(y_raw.min())
        y_model = (~np.isclose(y_raw.values, Y_CUTOFF)).astype(float) # 1 = above the lowest value (active)
        TARGET_TRANSFORM = f"binary: active = value above {Y_CUTOFF:.4g}"
    else:
        Y_CUTOFF = float(y_raw.median())
        y_model = (y_raw > Y_CUTOFF).astype(float).values
        TARGET_TRANSFORM = f"binary: active = value above median {Y_CUTOFF:.4g}"
        logging.warning(f" - No block of values at the minimum; classes split at the median")
    if min(y_model.sum(), len(y_model) - y_model.sum()) < N_SPLITS:
        raise RuntimeError(f"Too few samples in one class for {N_SPLITS}-fold CV")
    METRIC, CHANCE = "AUC", 0.5
else:
    if ENDPOINT_PLAN["transform"] == "log10":
        y_offset = float(y_raw[y_raw > 0].min() / 2) if (y_raw <= 0).any() else 0.0
        y_model = np.log10(y_raw + y_offset).values
        TARGET_TRANSFORM = f"log10(y + {y_offset:.4g})" if y_offset > 0 else "log10(y)"
    else:
        y_model = y_raw.values
        TARGET_TRANSFORM = "none"
    METRIC, CHANCE = "Q²", 0.0

# Dust data for this endpoint's samples: re-scaled on these samples, constant features dropped
X_endpoint, x_const_feats = r_scale(X_tox_vs_dust_prep.loc[common_samples])
logging.info(f" - Features constant within these samples (dropped): {len(x_const_feats)}")

# NaNs left in the dust data (models will fail on these too)
n_nan_x = X_endpoint.isna().sum().sum()
logging.info(f" - NaNs in X_endpoint: {n_nan_x} (in {X_endpoint.isna().any().sum()} features)")
if n_nan_x > 0:
    raise ValueError("NaNs in X_endpoint; set NAN_TO_ZERO = True in the Pt1 dust preprocessing cell")
minorsep()

END_OUT_DIR = (OUT_DIR / "endpoint_models" / ENDPOINT_SHORT.replace("/", "_"))
END_OUT_DIR.mkdir(parents=True, exist_ok=True)

logging.info(f" - Current endpoint: {ENDPOINT}")
logging.info(f" - Current endpoint short name: {ENDPOINT_SHORT}")
logging.info(f" - Task: {TASK} | target: {TARGET_TRANSFORM} | metric: {METRIC} (chance level = {CHANCE})")
if TASK == "classification":
    logging.info(f" - Active samples: {int(y_model.sum())} of {len(y_model)}")
logging.info(f" - Y shape [samples, tox endpoints]: {Y_endpoint.shape}")
logging.info(f" - X shape [samples, features]: {X_endpoint.shape}")
logging.info(f" - CV: {N_SPLITS} folds x {N_REPEATS} repeats | permutations: {N_PERM} | stability runs: {N_STABILITY_RUNS}")
minorsep()
logging.info(f"Y stats (raw): \n{Y_endpoint.describe()}")
minorsep()
logging.info(f"Y used for modeling: \n{pd.Series(y_model, name=ENDPOINT_SHORT).describe()}")

minorsep()
logging.info(f"Picked endpoint to model | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Univariate screen: which compounds go with higher endpoint values?
# Insert right after the "Endpoint selection" cell (uses ENDPOINT, ENDPOINT_SHORT, Y_endpoint, common_samples,
# END_OUT_DIR). Works for any endpoint and is independent of the Elastic Net / XGBoost cells.
# Three tests per feature, each with Benjamini-Hochberg (BH) correction across all screened features:
# - Spearman correlation with the endpoint (all samples; rank-based, so the log transform doesn't matter)
# - Mann-Whitney U, one-sided: concentration higher in the top samples than in the rest.
#   p-values from PERMUTATIONS of the top/rest labels (the standard normal approximation gives up to ~10x too
#   small p-values for sparsely detected features: many tied zeros + a small top group)
# - Fisher's exact, one-sided: detected more often in the top samples than in the rest
# Uses the raw dust data with its own (lower) detection threshold, so rare compounds present mainly in the
# most active samples are not removed by MIN_DETECT_FRAC.
majorsep()
logging.info(f"Univariate screen ({ENDPOINT_SHORT}) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

from scipy.stats import mannwhitneyu, hypergeom, t as t_dist

SCREEN_TOP_FRAC = 0.10 # top 10% of samples by endpoint value = "top" group (~16 of 164)
SCREEN_MIN_TOP_N = 8 # top group has at least this many samples
SCREEN_MIN_DETECTED = 5 # screen features detected (> 0) in >= 5 samples
FDR_LEVEL = 0.10 # Benjamini-Hochberg q-value below this = "hit"
SCREEN_SHOW_N = 20 # rows shown per table
SCREEN_MW_PERM = 200000 # permutations for Mann-Whitney p-values (smallest possible p = 1 / 200001)
SCREEN_MW_REFINE_P = 0.05 # permutation p only for features with approximate p below this (others can't be hits)

def bh_qvalues(p):
    """Benjamini-Hochberg q-values; NaN p-values stay NaN and are not counted as tests."""
    p = np.asarray(p, dtype=float)
    q = np.full(p.shape, np.nan)
    ok = ~np.isnan(p)
    m = int(ok.sum())
    if m == 0:
        return q
    p_ok = p[ok]
    order = np.argsort(p_ok)
    ranked = p_ok[order] * m / np.arange(1, m + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    q_ok = np.empty(m)
    q_ok[order] = np.minimum(ranked, 1.0)
    q[ok] = q_ok
    return q

# Data: endpoint values and raw dust data for the same samples
y_scr = Y_endpoint[ENDPOINT].astype(float).loc[common_samples]
X_raw_scr = X_tox_vs_dust.loc[common_samples].fillna(0)
det_scr = X_raw_scr > 0 # negative (below LOD) = not detected
neg_handling_scr = globals().get("NEG_HANDLING", "abs")
X_scr = X_raw_scr.abs() if neg_handling_scr == "abs" else X_raw_scr.clip(lower=0)
n_scr = len(y_scr)

# Feature selection for the screen
n_det_scr = det_scr.sum()
low_det = n_det_scr < SCREEN_MIN_DETECTED
constant = X_scr.nunique() <= 1
keep_scr = ~low_det & ~constant
X_scr, det_scr = X_scr.loc[:, keep_scr], det_scr.loc[:, keep_scr]

logging.info(f" - Samples: {n_scr}")
logging.info(f" - Features in raw dust data: {len(keep_scr):,}")
logging.info(f"   excluded, detected in < {SCREEN_MIN_DETECTED} samples: {int(low_det.sum()):,}")
logging.info(f"   excluded, constant: {int((constant & ~low_det).sum()):,}")
logging.info(f"   screened: {X_scr.shape[1]:,}")
if X_scr.shape[1] == 0:
    raise ValueError("No features left to screen; lower SCREEN_MIN_DETECTED")
lab_scr = (pd.Series(X_scr.columns, index=X_scr.columns).astype(str)
           .str.extract(r"_(UMU|VU)(?=_|$)")[0].fillna("other"))
logging.info(f"   screened per lab: {lab_scr.value_counts().to_dict()}")
minorsep()

# Top group
n_top_target = max(int(np.ceil(SCREEN_TOP_FRAC * n_scr)), SCREEN_MIN_TOP_N)
y_cut = y_scr.sort_values(ascending=False).iloc[n_top_target - 1]
is_top = (y_scr >= y_cut).values
top_rule = ">="
if int((~is_top).sum()) < 5: # cut-off falls on a large block of tied values (e.g., most samples at the lowest value)
    is_top = (y_scr > y_cut).values # top group = samples strictly above the tied value
    top_rule = ">"
    logging.warning(f" - Cut-off {y_cut:.4g} is shared by many samples: top group = samples with {ENDPOINT_SHORT} > {y_cut:.4g} "
                    f"({int(is_top.sum())} samples instead of the target {n_top_target}); screen results less reliable")
n_top, n_rest = int(is_top.sum()), int((~is_top).sum())
if n_rest < 5 or n_top < 3:
    raise ValueError(f"Top group cannot be formed (top {n_top}, rest {n_rest}); too many tied endpoint values")

# logging.info(f" - Top group: {n_top} samples with {ENDPOINT_SHORT} >= {y_cut:.4g} "
logging.info(f" - Top group: {n_top} samples with {ENDPOINT_SHORT} {top_rule} {y_cut:.4g} "
             f"(target {n_top_target}; more if tied at the cut-off)")
logging.info(f" - Rest: {n_rest} samples | median {y_scr[~is_top].median():.4g}, max {y_scr[~is_top].max():.4g}")
logging.info(f" - Top median / rest median: {y_scr[is_top].median() / max(y_scr[~is_top].median(), 1e-12):.1f}x")
minorsep()
logging.info(f"Top samples:")
display(y_scr[is_top].sort_values(ascending=False).rename(ENDPOINT).to_frame())
minorsep()

Xv = X_scr.values
Dv = det_scr.values

# Spearman correlation (vectorized; same p-values as scipy.stats.spearmanr)
Xr = rankdata(Xv, axis=0)
yr = rankdata(y_scr.values)
Xc, yc = Xr - Xr.mean(axis=0), yr - yr.mean()
denom = np.sqrt((Xc ** 2).sum(axis=0) * (yc ** 2).sum())
rho = np.divide(Xc.T @ yc, denom, out=np.full(Xv.shape[1], np.nan), where=denom > 0)
rho_c = np.clip(rho, -0.999999, 0.999999)
p_spearman = 2 * t_dist.sf(np.abs(rho_c * np.sqrt((n_scr - 2) / (1 - rho_c ** 2))), n_scr - 2)

# Mann-Whitney U, one-sided (top > rest)
with warnings.catch_warnings(), np.errstate(all="ignore"):
    warnings.simplefilter("ignore")
    mw = mannwhitneyu(Xv[is_top], Xv[~is_top], alternative="greater", axis=0, method="asymptotic")
p_mw_approx = np.asarray(mw.pvalue, dtype=float)
auc_top = np.asarray(mw.statistic, dtype=float) / (n_top * n_rest) # P(top sample > rest sample); 0.5 = no difference

# Permutation p-values: rank sum of the top group vs. rank sums of random groups of the same size
p_mw = p_mw_approx.copy()
refine_idx = np.where(p_mw_approx < SCREEN_MW_REFINE_P)[0]
logging.info(f" - Mann-Whitney: permutation p-values ({SCREEN_MW_PERM:,} permutations) for {len(refine_idx):,} features "
             f"with approximate p < {SCREEN_MW_REFINE_P}")
minorsep()
if len(refine_idx) > 0:
    R_mw = rankdata(Xv[:, refine_idx], axis=0).astype(np.float32) # ranks per feature, ties averaged
    rs_obs = R_mw[is_top].sum(axis=0)
    n_exceed = np.zeros(len(refine_idx))
    rng_mw = np.random.default_rng(GLOBAL_SEED)
    chunk_mw = 5000
    bar_mw = tqdm(total=SCREEN_MW_PERM, desc="Mann-Whitney permutations")
    done_mw = 0
    while done_mw < SCREEN_MW_PERM:
        m = min(chunk_mw, SCREEN_MW_PERM - done_mw)
        top_idx = np.argpartition(rng_mw.random((m, n_scr)), n_top, axis=1)[:, :n_top] # random top groups
        P = np.zeros((m, n_scr), dtype=np.float32)
        np.put_along_axis(P, top_idx, 1.0, axis=1)
        n_exceed += ((P @ R_mw) >= rs_obs - 1e-3).sum(axis=0)
        done_mw += m
        bar_mw.update(m)
    bar_mw.close()
    p_mw[refine_idx] = (n_exceed + 1) / (SCREEN_MW_PERM + 1)
    minorsep()
    logging.info(f" - Smallest approximate p: {p_mw_approx[refine_idx].min():.2g} -> permutation p: {p_mw[refine_idx].min():.2g}")

# Fisher's exact, one-sided (detected more often in top), via the hypergeometric distribution
det_top = Dv[is_top].sum(axis=0)
det_all = Dv.sum(axis=0)
p_fisher = hypergeom.sf(det_top - 1, n_scr, det_all, n_top)

screen_df = pd.DataFrame({
    "ML_feature_id": X_scr.columns,
    "lab": lab_scr.values,
    "n_detected": det_all,
    "det_frac_top": det_top / n_top,
    "det_frac_rest": (det_all - det_top) / n_rest,
    "median_top": np.median(Xv[is_top], axis=0),
    "median_rest": np.median(Xv[~is_top], axis=0),
    "spearman_rho": rho,
    "p_spearman": p_spearman,
    "mw_auc_top": auc_top,
    "p_mannwhitney": p_mw,
    "p_mannwhitney_approx": p_mw_approx,
    "p_fisher": p_fisher,
})
screen_df["q_spearman"] = bh_qvalues(screen_df["p_spearman"])
screen_df["q_mannwhitney"] = bh_qvalues(screen_df["p_mannwhitney"])
screen_df["q_fisher"] = bh_qvalues(screen_df["p_fisher"])
# Spearman is two-sided: a hit counts only if the correlation is positive (higher conc. -> higher endpoint)
screen_df["hit_spearman"] = (screen_df["q_spearman"] < FDR_LEVEL) & (screen_df["spearman_rho"] > 0)
screen_df["hit_mannwhitney"] = screen_df["q_mannwhitney"] < FDR_LEVEL
screen_df["hit_fisher"] = screen_df["q_fisher"] < FDR_LEVEL
screen_df["n_tests_hit"] = screen_df[["hit_spearman", "hit_mannwhitney", "hit_fisher"]].sum(axis=1)

# QC: are there more small p-values than expected by chance?
minorsep()
logging.info(f"QC: p-value distribution (pure noise = flat; expected count per bin shown)")
minorsep()
p_bins = [0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
qc_rows = {}
for col in ["p_spearman", "p_mannwhitney", "p_fisher"]:
    pv = screen_df[col].dropna()
    qc_rows[col] = pd.cut(pv, p_bins, include_lowest=True).value_counts().sort_index().values
n_tests_scr = int(screen_df["p_spearman"].notna().sum())
qc_df = pd.DataFrame(qc_rows, index=[f"{a}-{b}" for a, b in zip(p_bins[:-1], p_bins[1:])])
qc_df["expected_if_noise"] = (np.diff(p_bins) * n_tests_scr).round(0).astype(int)
display(qc_df)
logging.info(f" - Note: Fisher p-values are conservative for rarely detected features (fewer small p-values "
             f"than 'expected' does not mean anything for Fisher)")
minorsep()

logging.info(f"Summary (Benjamini-Hochberg correction, FDR {FDR_LEVEL:.0%}):")
for test, pcol, qcol, hcol in [("Spearman (positive)", "p_spearman", "q_spearman", "hit_spearman"),
                               ("Mann-Whitney top > rest", "p_mannwhitney", "q_mannwhitney", "hit_mannwhitney"),
                               ("Fisher detected more in top", "p_fisher", "q_fisher", "hit_fisher")]:
    n_t = int(screen_df[pcol].notna().sum())
    n_p05 = int((screen_df[pcol] < 0.05).sum())
    logging.info(f" - {test}: {n_t:,} tests | p < 0.05: {n_p05:,} (expected by chance: ~{0.05 * n_t:,.0f}) | "
                 f"hits (q < {FDR_LEVEL}): {int(screen_df[hcol].sum()):,} | smallest q: {screen_df[qcol].min():.3g}")
logging.info(f" - Features that are hits in >= 2 tests: {int((screen_df['n_tests_hit'] >= 2).sum()):,}")

if screen_df["n_tests_hit"].sum() == 0:
    minorsep()
    logging.warning(f" !!! No feature passes Benjamini-Hochberg correction in any test: \nNo detectable compound-activity link "
                    f"for {ENDPOINT_SHORT} at FDR {FDR_LEVEL:.0%}")
minorsep()

# Metadata: always taken from the FULL classified metadata (feat_meta_class), so this cell does not depend
# on which version of add_meta() is in memory
def screen_meta(df):
    if "feat_meta_class" not in globals():
        logging.warning(" - feat_meta_class not loaded: run the 'Load preprocessed data' cell first")
        return df
    meta = feat_meta_class.copy()
    meta["ML_feature_id"] = meta["ML_feature_id"].astype(str).str.strip()
    meta = meta.drop_duplicates("ML_feature_id")
    meta = meta[["ML_feature_id"] + [c for c in meta.columns if c not in df.columns]]
    out = df.copy()
    out["ML_feature_id"] = out["ML_feature_id"].astype(str).str.strip()
    return out.merge(meta, on="ML_feature_id", how="left")

# Metadata QC
logging.info(f"Metadata QC:")
if "feat_meta_class" in globals():
    meta_ids_scr = feat_meta_class["ML_feature_id"].astype(str).str.strip()
    logging.info(f" - feat_meta_class: {len(feat_meta_class):,} rows, {meta_ids_scr.nunique():,} unique IDs")
    screen_ids = screen_df["ML_feature_id"].astype(str).str.strip()
    unmatched = screen_ids[~screen_ids.isin(set(meta_ids_scr))]
    logging.info(f" - Screened features found in feat_meta_class: {len(screen_ids) - len(unmatched):,} of {len(screen_ids):,}")
    if len(unmatched) > 0:
        logging.warning(f" - Not found (first 10, repr shows hidden characters): {[repr(x) for x in unmatched.head(10)]}")
    meta_check = screen_meta(screen_df[["ML_feature_id"]])
    if "feature_name" in meta_check.columns:
        n_no_name = int(meta_check["feature_name"].isna().sum())
        logging.info(f" - Screened features with empty feature_name after merge: {n_no_name:,}")
minorsep()

# Tables
show_cols = ["ML_feature_id", "feature_name", "class", "lab", "n_detected", "det_frac_top", "det_frac_rest",
             "median_top", "median_rest", "spearman_rho", "q_spearman", "mw_auc_top", "q_mannwhitney",
             "q_fisher", "n_tests_hit"]

def _show(df_sorted, title):
    logging.info(title)
    out = screen_meta(df_sorted.head(SCREEN_SHOW_N))
    with pd.option_context("display.max_colwidth", 40):
        display(out[[c for c in show_cols if c in out.columns]].round(4))
    minorsep()

_show(screen_df[screen_df["spearman_rho"] > 0].sort_values("p_spearman"), f"Top {SCREEN_SHOW_N} by Spearman (positive correlations):")
_show(screen_df.sort_values("p_mannwhitney"), f"Top {SCREEN_SHOW_N} by Mann-Whitney (higher in top samples):")
_show(screen_df.sort_values("p_fisher"), f"Top {SCREEN_SHOW_N} by Fisher (detected more often in top samples):")

if (screen_df["n_tests_hit"] >= 1).any():
    _show(screen_df[screen_df["n_tests_hit"] >= 1].sort_values(["n_tests_hit", "q_mannwhitney"], ascending=[False, True]),
          f"All hits (q < {FDR_LEVEL} in >= 1 test), most consistent first:")

# Save
screen_df = screen_df.sort_values("p_mannwhitney").reset_index(drop=True)
screen_name = f"univariate_screen_{ENDPOINT_SHORT}"
screen_df.to_csv(END_OUT_DIR / f"{screen_name}.csv", index=False)
screen_meta(screen_df).to_csv(END_OUT_DIR / f"{screen_name}_w_metadata.csv", index=False)
logging.info(f"Saved: {END_OUT_DIR / f'{screen_name}_w_metadata.csv'}")

minorsep()
logging.info(f"Done univariate screen ({ENDPOINT_SHORT}) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Train Elastic Net
majorsep()
logging.info(f"Training Elastic Net | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# alpha grid from the data (alpha_max = smallest alpha giving all-zero coefficients, down 3 orders
# of magnitude). The fixed grid 0.0001-10 was too narrow: andro picked alpha = 10 = the grid edge
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9]
N_ALPHAS = 60
y_centered = y_model - y_model.mean()
alpha_max = np.max(np.abs(X_endpoint.values.T @ y_centered)) / (len(y_model) * min(L1_RATIOS))
alphas = np.logspace(np.log10(alpha_max) - 3, np.log10(alpha_max), N_ALPHAS)

enet_cv = ElasticNetCV(
    l1_ratio=L1_RATIOS, # wider range; CV picks between grouped (low) and sparse (high) selection
    alphas=alphas,
    cv=make_cv(TASK, N_SPLITS, GLOBAL_SEED), # shuffled folds (cv=5 used unshuffled, sorted-by-name blocks)
    max_iter=10000,
    n_jobs=-1,
    verbose=0
)

enet_cv.fit(X_endpoint, y_model) # transformed target (log10 or 0/1)

logging.info(f" - Best alpha ({alphas.min():.4g}—{alphas.max():.4g}): {enet_cv.alpha_:.4g}")
logging.info(f" - Best l1_ratio: {enet_cv.l1_ratio_}")

# Warn if the best alpha sits on the edge of the grid
if np.isclose(enet_cv.alpha_, alphas.max()):
    logging.warning(f" !!! Best alpha = largest alpha (all or nearly all coefficients zero): no linear signal found")
elif np.isclose(enet_cv.alpha_, alphas.min()):
    logging.warning(f" !!! Best alpha = smallest alpha in the grid; consider extending it downwards")

minorsep()
logging.info(f"Done training Elastic Net | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Elastic Net performance
# - Q² (regression) or AUC (classification), on the modeled scale (log10 or 0/1)
# - Repeated CV: one pooled out-of-fold metric per repeat (N_SPLITS folds x N_REPEATS repeats)
#   The old per-fold R² (on ~33 test samples) was very unstable (SD 1.6) and a single extreme sample could dominate a fold
# - Nested CV: alpha is re-tuned inside every training fold (l1_ratio fixed to the best one), so the test folds
#   and the real y don't leak into the result. The old fixed-alpha CV/permutation test was optimistic
majorsep()
logging.info(f"Elastic Net performance | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Elastic Net model with the tuned settings (used for stability selection)
enet_model = ElasticNet(
    alpha=enet_cv.alpha_,
    l1_ratio=enet_cv.l1_ratio_,
    max_iter=10000
)

# Elastic Net that re-tunes alpha on each training fold (used for CV and permutation tests)
alphas_eval = alphas[::3] # 20 alphas over the same range
enet_eval_model = make_tuned_enet(TASK, enet_cv.l1_ratio_, alphas_eval, GLOBAL_SEED)

# Training metric (fit on all samples; optimistic by definition)
enet_train_metric = score_metric(TASK, y_model, enet_cv.predict(X_endpoint))
logging.info(f" - Training {METRIC} completed")
minorsep()

# Repeated CV metric
enet_cv_scores = repeated_cv_metric(enet_eval_model, X_endpoint, y_model, TASK, N_SPLITS, N_REPEATS, GLOBAL_SEED)
logging.info(f" - Repeated CV completed")
minorsep()

logging.info(f" - Training {METRIC}: {enet_train_metric:.3f}")
logging.info(f" - CV {METRIC} mean ({N_SPLITS} folds x {N_REPEATS} repeats): {enet_cv_scores.mean():.3f}")
logging.info(f" - CV {METRIC} SD (between repeats): {enet_cv_scores.std():.3f}")
logging.info(f" - Chance level: {CHANCE}")

minorsep()
logging.info(f"Done Elastic Net performance | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Permutation test - Elastic Net
# Same pooled CV metric as above, compared with N_PERM runs on shuffled y
majorsep()
logging.info(f"Running Elastic Net permutation test | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

enet_perm_score, enet_perm_scores, enet_pvalue = permutation_test_metric(
    enet_eval_model, X_endpoint, y_model, TASK, N_SPLITS, N_PERM, GLOBAL_SEED
)

# Does the linear model carry real signal?
ENET_HAS_SIGNAL = bool((enet_cv_scores.mean() > CHANCE) and (enet_pvalue < 0.05))

minorsep()
logging.info(f" - Elastic Net observed {METRIC}: {enet_perm_score:.3f}")
logging.info(f" - Permuted {METRIC}: mean {enet_perm_scores.mean():.3f}, 95th percentile {np.percentile(enet_perm_scores, 95):.3f}")
logging.info(f" - Elastic Net permutation p-value: {enet_pvalue:.4f}")
logging.info(f" - Elastic Net has signal (CV {METRIC} > {CHANCE} and p < 0.05): {ENET_HAS_SIGNAL}")
if not ENET_HAS_SIGNAL:
    minorsep()
    logging.warning(f" !!! No reliable Elastic Net signal: compound rankings for this endpoint are exploratory only")

minorsep()
logging.info(f"Done Elastic Net permutation test | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Elastic Net coefficients, importance, sparsity QC & outputs
# Merged 5 old cells (coefficients, importance score, sparsity QC, save, top-20 preview)
# - Keeps the coefficient sign: + = higher concentration goes with higher endpoint value (more toxic)
# - Uses feat_meta_dust_masked (old 'feature_metadata_rnm' did not exist → NameError)
# - Removed duplicate outputs (coefficients_selected and features_selected were the same table)
majorsep()
logging.info(f"Elastic Net coefficients & outputs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

coef_matrix = pd.DataFrame({
    "ML_feature_id": X_endpoint.columns,
    "enet_coef": enet_cv.coef_,
})
coef_matrix["enet_importance"] = coef_matrix["enet_coef"].abs()

enet_rank = (
    coef_matrix
    .sort_values("enet_importance", ascending=False)
    .reset_index(drop=True)
)

enet_nonzero_features = int((coef_matrix["enet_importance"] > 0).sum())
enet_n_positive = int((coef_matrix["enet_coef"] > 0).sum())

logging.info(f" - Total compounds: {len(coef_matrix):,}")
logging.info(f" - Selected compounds (coef ≠ 0): {enet_nonzero_features:,} ({100 * enet_nonzero_features / len(coef_matrix):.2f}%)")
logging.info(f"   of which positive (higher conc. → higher endpoint): {enet_n_positive:,}")
minorsep()

if enet_nonzero_features == 0:
    logging.warning(f" !!! No Elastic Net signal detected (all coefficients zero)")
    minorsep()

# Save
save_with_meta(enet_rank, "elastic_net_coefficients")
save_with_meta(enet_rank[enet_rank["enet_importance"] > 0], "elastic_net_coefficients_selected")

logging.info(f"Top 20 Elastic Net compounds:")
display(add_meta(enet_rank.head(20)))

minorsep()
logging.info(f"Done Elastic Net coefficients & outputs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Stability selection
# Merged 6 old cells (setup, loop, checks, scores, stable set, QC)
# - Runs in parallel (much faster)
# - Classification: each subsample keeps the active/inactive ratio
# - stability_score = fraction of runs in which a feature got a non-zero coefficient
majorsep()
logging.info(f"Running stability selection | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

logging.info(f" - Stability runs: {N_STABILITY_RUNS}")
logging.info(f" - Subsample fraction: {SUBSAMPLE_FRACTION}")
logging.info(f" - Selection threshold: {STABILITY_THRESHOLD}")
minorsep()

def _stability_run(X_arr, y_arr, task, frac, alpha, l1_ratio, seed):
    rng = np.random.default_rng(seed)
    if task == "classification": # same class ratio in every subsample
        idx = np.concatenate([
            rng.choice(np.where(y_arr == c)[0], max(1, int(round(frac * (y_arr == c).sum()))), replace=False)
            for c in np.unique(y_arr)
        ])
    else:
        idx = rng.choice(len(y_arr), int(frac * len(y_arr)), replace=False)
    m = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000).fit(X_arr[idx], y_arr[idx])
    return np.abs(m.coef_) > 1e-8 # tiny coefficients = numerical noise

stability_selected = Parallel(n_jobs=-1)(
    delayed(_stability_run)(X_endpoint.values, y_model, TASK, SUBSAMPLE_FRACTION,
                            enet_cv.alpha_, enet_cv.l1_ratio_, GLOBAL_SEED + i)
    for i in tqdm(range(N_STABILITY_RUNS), desc="Stability selection")
)
stability_counts = pd.Series(np.sum(stability_selected, axis=0), index=X_endpoint.columns)
stability_score = stability_counts / N_STABILITY_RUNS

stability_rank = (
    pd.DataFrame({"ML_feature_id": stability_score.index, "stability_score": stability_score.values})
    .sort_values("stability_score", ascending=False)
    .reset_index(drop=True)
)

stable_features = stability_score[stability_score >= STABILITY_THRESHOLD].index

minorsep()
logging.info(f" - Features selected per run: mean {np.mean([s.sum() for s in stability_selected]):.0f}")
logging.info(f" - Features with stability > 0: {(stability_score > 0).sum():,}")
logging.info(f" - Stable features (score ≥ {STABILITY_THRESHOLD}): {len(stable_features):,} "
             f"({100 * len(stable_features) / X_endpoint.shape[1]:.2f}%)")
logging.info(f" - High-confidence features (score ≥ {HIGH_CONF_THRESHOLD}): {(stability_score >= HIGH_CONF_THRESHOLD).sum():,}")
minorsep()

display(stability_rank.head(20))
minorsep()

# Stability distribution QC
stability_bins = {
    "0.0": (stability_score == 0.0).sum(),
    "0.0–0.2": ((stability_score > 0.0) & (stability_score < 0.2)).sum(),
    "0.2–0.4": ((stability_score >= 0.2) & (stability_score < 0.4)).sum(),
    "0.4–0.6": ((stability_score >= 0.4) & (stability_score < 0.6)).sum(),
    "0.6–0.8": ((stability_score >= 0.6) & (stability_score < 0.8)).sum(),
    "0.8–1.0": ((stability_score >= 0.8) & (stability_score < 1.0)).sum(),
    "1.0": (stability_score == 1.0).sum()
}
display(pd.DataFrame(list(stability_bins.items()), columns=["stability_range", "n_features"]))

minorsep()
logging.info(f"Done stability selection | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Save stability outputs
# Shorter (save_with_meta helper); thresholds from settings instead of hard-coded 0.6/0.8
majorsep()
logging.info(f"Saving stability outputs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

save_with_meta(stability_rank, "stability_rank")
save_with_meta(stability_rank[stability_rank["stability_score"] >= STABILITY_THRESHOLD], "stable_features")
save_with_meta(stability_rank[stability_rank["stability_score"] >= HIGH_CONF_THRESHOLD], "high_confidence_features")

logging.info(f"Done saving stability outputs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Prepare XGBoost input matrix & configuration
# Merged with the configuration cell
# - XGBoost gets at most XGB_MAX_FEATURES stable features (andro had 2,224 for 164 samples = heavy overfitting);
#   ordered by stability score, then |Elastic Net coefficient|
# - Model a bit more regularized (colsample 0.5, min_child_weight 3); binary:logistic for classification
majorsep()
logging.info(f"Preparing XGBoost input matrix | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

xgb_candidates = (
    stability_rank.merge(coef_matrix[["ML_feature_id", "enet_importance"]], on="ML_feature_id", how="left")
    .sort_values(["stability_score", "enet_importance"], ascending=False)
)
high_conf_features = xgb_candidates.loc[xgb_candidates["stability_score"] >= STABILITY_THRESHOLD, "ML_feature_id"]

if len(high_conf_features) < 10:
    logging.warning(f" - Only {len(high_conf_features)} stable features; using the 10 most stable instead")
    high_conf_features = xgb_candidates.loc[xgb_candidates["stability_score"] > 0, "ML_feature_id"].head(10)

high_conf_features = high_conf_features.head(XGB_MAX_FEATURES).tolist()
X_xgb = X_endpoint[high_conf_features].copy()

# XGBoost steps run only if stability selection selected something
XGB_RAN = len(high_conf_features) > 0
if not XGB_RAN:
    logging.warning(f" !!! No feature was ever selected in stability selection (no linear signal): "
                    f"XGBoost/SHAP steps are skipped for this endpoint; outputs are still saved")

logging.info(f" - X_xgb shape [samples, features]: {X_xgb.shape}")
logging.info(f" - y shape [samples]: {y_model.shape}")
minorsep()

xgb_params = dict(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.5,
    min_child_weight=3,
    reg_lambda=1.0,
    objective="binary:logistic" if TASK == "classification" else "reg:squarederror",
    random_state=GLOBAL_SEED,
    n_jobs=1 # CV runs folds in parallel instead
)

display(pd.DataFrame([xgb_params]))

minorsep()
logging.info(f"Done preparing XGBoost input matrix | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# XGBoost cross-validated performance & permutation test
# - Bug fixed (data leakage): the old XGBoost CV used features chosen by stability selection on ALL samples,
#  so the test folds had already "seen" the features → optimistic Q², and the permutation test was invalid
#  (features chosen with the real y were kept for shuffled y; hence p = 0.0099 with Q² = -0.30 for andro)
# - Now feature selection (Elastic Net with alpha re-tuned, top XGB_MAX_FEATURES) is re-fitted INSIDE every training fold
majorsep()
logging.info(f"Running XGBoost CV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

if XGB_RAN:
    xgb_cv_pipe = Pipeline([
        ("select", ENetTopKSelector(enet=make_tuned_enet(TASK, enet_cv.l1_ratio_, alphas[::3], GLOBAL_SEED), k=X_xgb.shape[1])),
        ("xgb", XGBRegressor(**xgb_params)),
    ])
    xgb_cv_scores = repeated_cv_metric(xgb_cv_pipe, X_endpoint, y_model, TASK, N_SPLITS, N_REPEATS, GLOBAL_SEED)
else:
    logging.info(f" - Skipped (no stable features)")
    xgb_cv_scores = np.array([np.nan])

xgb_cv_results = pd.DataFrame({
    "endpoint": [ENDPOINT],
    "metric": [METRIC],
    "CV_mean": [xgb_cv_scores.mean()],
    "CV_sd": [xgb_cv_scores.std()]
})
display(xgb_cv_results)
minorsep()

if XGB_RAN:
    logging.info(f"Running XGBoost permutation test ({N_PERM} permutations) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    minorsep()
    xgb_perm_score, xgb_perm_scores, xgb_pvalue = permutation_test_metric(
        xgb_cv_pipe, X_endpoint, y_model, TASK, N_SPLITS, N_PERM, GLOBAL_SEED
    )
else:
    xgb_perm_score, xgb_perm_scores, xgb_pvalue = np.nan, np.array([]), np.nan

XGB_HAS_SIGNAL = bool(XGB_RAN and (xgb_cv_scores.mean() > CHANCE) and (xgb_pvalue < 0.05))

minorsep()
logging.info(f" - XGBoost CV {METRIC} mean ({N_SPLITS} folds x {N_REPEATS} repeats): {xgb_cv_scores.mean():.3f}")
logging.info(f" - XGBoost CV {METRIC} SD: {xgb_cv_scores.std():.3f}")
logging.info(f" - XGBoost observed {METRIC}: {xgb_perm_score:.3f}")
logging.info(f" - XGBoost permutation p-value: {xgb_pvalue:.4f}")
logging.info(f" - XGBoost has signal: {XGB_HAS_SIGNAL}")

minorsep()
logging.info(f"Done XGBoost CV & permutation test | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Train final XGBoost model and compute SHAP
# Adds SHAP direction (+ = higher concentration pushes the prediction up);
# falls back to XGBoost's built-in SHAP if the shap package fails (shap/xgboost version clashes)
majorsep()
logging.info(f"Computing SHAP values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

if XGB_RAN:
    xgb_model = XGBRegressor(**{**xgb_params, "n_jobs": -1})
    xgb_model.fit(X_xgb, y_model)

    try:
        shap_values = np.asarray(shap.TreeExplainer(xgb_model).shap_values(X_xgb))
    except Exception as e:
        logging.warning(f" - shap package failed ({type(e).__name__}: {e}); using XGBoost built-in SHAP values")
        shap_values = xgb_model.get_booster().predict(xgb.DMatrix(X_xgb), pred_contribs=True)[:, :-1]

    # direction = rank correlation between feature value and its SHAP value
    x_ranks = rankdata(X_xgb.values, axis=0)
    s_ranks = rankdata(shap_values, axis=0)
    x_c, s_c = x_ranks - x_ranks.mean(axis=0), s_ranks - s_ranks.mean(axis=0)
    denom = np.sqrt((x_c ** 2).sum(axis=0) * (s_c ** 2).sum(axis=0))
    shap_direction = np.divide((x_c * s_c).sum(axis=0), denom, out=np.zeros(X_xgb.shape[1]), where=denom > 0)
else:
    logging.info(f" - Skipped (no stable features)")
    shap_values = np.zeros((len(X_xgb), 0))
    shap_direction = np.zeros(0)

endpoint_shap_importance = pd.DataFrame({
    "ML_feature_id": X_xgb.columns,
    "shap_importance": np.abs(shap_values).mean(axis=0),
    "shap_direction": shap_direction,
})

# SHAP ranking (merged from the old "SHAP ranking" cell)
shap_rank = (
    endpoint_shap_importance
    .sort_values("shap_importance", ascending=False)
    .reset_index(drop=True)
)

display(shap_rank.head(20))

minorsep()
logging.info(f" - SHAP table rows: {len(endpoint_shap_importance):,}")
logging.info(f" - Features used by the final XGBoost model (SHAP > 0): {(endpoint_shap_importance['shap_importance'] > 0).sum():,}")

minorsep()
logging.info(f"Done computing SHAP values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Final Stability-Adjusted Compound Index (FSACI)
# New FSACI = stability_score x effect_score
# - stability_score: fraction of stability runs selecting the compound (0-1): is it selected consistently?
# - effect_score: how strongly it drives the prediction, as percentile ranks among non-zero compounds (0-1):
#   - Elastic Net |coef| and XGBoost SHAP, weighted W_ENET / W_SHAP
#   - SHAP is only used if XGBoost has signal (otherwise its importances are noise)
#   - Percentile ranks are used instead of min-max scaling, so one extreme value doesn't squash all others
#   - Multiplying means a compound must be BOTH stable AND influential to rank high
#   - direction: + = higher concentration goes with higher endpoint value (more toxic / active);
#     from the Elastic Net sign, else from SHAP. With PRIORITIZE_POSITIVE_ONLY, compounds with a negative
#     direction get FSACI = 0 (kept in FSACI_any_direction)
majorsep()
logging.info(f"Computing FSACI (final stability-adjusted compound index) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

W_ENET, W_SHAP = 0.5, 0.5
USE_SHAP_IN_FSACI = XGB_HAS_SIGNAL

def pct_rank_nonzero(s):
    out = pd.Series(0.0, index=s.index)
    nz = s > 0
    if nz.any():
        out[nz] = s[nz].rank(pct=True)
    return out

fsaci_rank = (
    coef_matrix[["ML_feature_id", "enet_coef", "enet_importance"]]
    .merge(stability_rank, on="ML_feature_id", how="left")
    .merge(endpoint_shap_importance, on="ML_feature_id", how="left")
)
fsaci_rank[["stability_score", "shap_importance", "shap_direction"]] = (
    fsaci_rank[["stability_score", "shap_importance", "shap_direction"]].fillna(0)
)
fsaci_rank["in_xgboost"] = fsaci_rank["ML_feature_id"].isin(X_xgb.columns)

fsaci_rank["enet_score"] = pct_rank_nonzero(fsaci_rank["enet_importance"])
fsaci_rank["shap_score"] = pct_rank_nonzero(fsaci_rank["shap_importance"])

if USE_SHAP_IN_FSACI:
    fsaci_rank["effect_score"] = (W_ENET * fsaci_rank["enet_score"] + W_SHAP * fsaci_rank["shap_score"]) / (W_ENET + W_SHAP)
else:
    fsaci_rank["effect_score"] = fsaci_rank["enet_score"]
    logging.info(f" - XGBoost has no signal: SHAP not used in FSACI (Elastic Net only)")
    minorsep()

fsaci_rank["direction"] = np.where(
    fsaci_rank["enet_coef"] != 0, np.sign(fsaci_rank["enet_coef"]), np.sign(fsaci_rank["shap_direction"])
).astype(int)

fsaci_rank["FSACI_any_direction"] = fsaci_rank["stability_score"] * fsaci_rank["effect_score"]
if PRIORITIZE_POSITIVE_ONLY:
    fsaci_rank["FSACI"] = np.where(fsaci_rank["direction"] > 0, fsaci_rank["FSACI_any_direction"], 0.0)
else:
    fsaci_rank["FSACI"] = fsaci_rank["FSACI_any_direction"]

fsaci_rank = (
    fsaci_rank
    .sort_values(["FSACI", "FSACI_any_direction", "stability_score"], ascending=False)
    .reset_index(drop=True)
)
fsaci_rank["FSACI_rank"] = np.arange(1, len(fsaci_rank) + 1)

display(fsaci_rank.head(20))

# QC
minorsep()
logging.info(f" - FSACI computed for {len(fsaci_rank):,} features")
logging.info(f" - Features with FSACI > 0: {(fsaci_rank['FSACI'] > 0).sum():,}")
logging.info(f" - Features with stability > 0: {(fsaci_rank['stability_score'] > 0).sum():,}")
logging.info(f" - Features with stability ≥ {STABILITY_THRESHOLD}: {(fsaci_rank['stability_score'] >= STABILITY_THRESHOLD).sum():,}")
logging.info(f" - Max FSACI score: {fsaci_rank['FSACI'].max():.3f}")
logging.info(f" - Top FSACI feature: {fsaci_rank.iloc[0]['ML_feature_id']} "
             f"(FSACI {fsaci_rank.iloc[0]['FSACI']:.3f}, stability {fsaci_rank.iloc[0]['stability_score']:.2f})")
MODEL_HAS_SIGNAL = ENET_HAS_SIGNAL or XGB_HAS_SIGNAL
if not MODEL_HAS_SIGNAL:
    minorsep()
    logging.warning(f" !!! Neither model has signal for {ENDPOINT_SHORT}: ranking is exploratory and excluded "
                    f"from the cross-endpoint summary by default")

minorsep()
logging.info(f"Done computing FSACI | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Save model outputs (XGBoost, SHAP, FSACI)
majorsep()
logging.info(f"Saving XGBoost & FSACI outputs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

xgb_cv_results.to_csv(END_OUT_DIR / "xgboost_cv_results.csv", index=False)
save_with_meta(shap_rank, "shap_rank")

# Full FSACI ranking (+ metadata, key columns first)
save_with_meta(fsaci_rank, "fsaci_rank")
fsaci_rank_meta = add_meta(fsaci_rank)
preferred_cols = ["FSACI_rank", "ML_feature_id", "feature_name", "class", "inchikey", "formula", "rt1_(min)", "rt2_(sec)",
                  "FSACI", "direction", "stability_score", "enet_coef", "shap_importance", "shap_direction"]
first_cols = [c for c in preferred_cols if c in fsaci_rank_meta.columns]
fsaci_rank_meta = fsaci_rank_meta[first_cols + [c for c in fsaci_rank_meta.columns if c not in first_cols]]
fsaci_rank_meta.to_csv(END_OUT_DIR / "fsaci_rank_w_metadata.csv", index=False)

# Top N FSACI compounds
fsaci_top = fsaci_rank_meta[fsaci_rank_meta["FSACI"] > 0].head(TOP_N_FSACI)
fsaci_top.to_csv(END_OUT_DIR / f"fsaci_top{TOP_N_FSACI}_w_metadata.csv", index=False)
logging.info(f" - Top FSACI compounds (FSACI > 0, max {TOP_N_FSACI}): {len(fsaci_top):,}")

# Top N FSACI + stable (stability ≥ STABILITY_THRESHOLD)
fsaci_top_highconf = fsaci_top[fsaci_top["stability_score"] >= STABILITY_THRESHOLD]
fsaci_top_highconf.to_csv(END_OUT_DIR / f"fsaci_top{TOP_N_FSACI}_stable_w_metadata.csv", index=False)
logging.info(f" - ... of which stable (≥ {STABILITY_THRESHOLD}): {len(fsaci_top_highconf):,}")
minorsep()

logging.info(f"FSACI outputs (top 30):")
display(fsaci_top.head(30))

# Endpoint ranking file for the cross-endpoint summary
endpoint_export = fsaci_rank[["ML_feature_id", "FSACI", "FSACI_rank", "direction"]].copy()
outfile = END_OUT_DIR / f"FSACI_{ENDPOINT_SHORT.replace('/', '_')}.csv"
endpoint_export.to_csv(outfile, index=False)

minorsep()
logging.info(f" - Saved endpoint ranking: \n   {outfile}")

minorsep()
logging.info(f"Done saving XGBoost & FSACI outputs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Endpoint statistics summary
# Oe table (the earlier Elastic Net + stability table was a repeat); metric names follow the task
majorsep()
logging.info(f"Building endpoint statistics table | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

y_stat = Y_endpoint.iloc[:, 0]

endpoint_stats = pd.DataFrame({
    # Endpoint
    "Endpoint": [ENDPOINT_SHORT],
    "Task": [TASK],
    "Target": [TARGET_TRANSFORM],
    "Metric": [METRIC],
    "Chance level": [CHANCE],
    "Endpoint count": [len(Y_endpoint)],
    "N active (classification)": [int(y_model.sum()) if TASK == "classification" else np.nan],
    "Endpoint mean": [float(y_stat.mean())],
    "Endpoint median": [float(y_stat.median())],
    "Endpoint SD": [float(y_stat.std())],
    "Endpoint min": [float(y_stat.min())],
    "Endpoint max": [float(y_stat.max())],
    "N unique values": [y_stat.nunique()],
    "% at most common value": [100 * ENDPOINT_PLAN["frac_mode"]],
    "Total features": [X_endpoint.shape[1]],

    # Elastic Net
    "Elastic Net best alpha": [enet_cv.alpha_],
    "Elastic Net best l1_ratio": [enet_cv.l1_ratio_],
    "Elastic Net training metric": [enet_train_metric],
    "Elastic Net CV metric mean": [enet_cv_scores.mean()],
    "Elastic Net CV metric SD": [enet_cv_scores.std()],
    "Elastic Net permutation observed": [enet_perm_score],
    "Elastic Net permutation p-value": [enet_pvalue],
    "Elastic Net signal": ["Yes" if ENET_HAS_SIGNAL else "No"],
    "Elastic Net selected compounds (≠ 0)": [enet_nonzero_features],
    "Elastic Net selected positive": [enet_n_positive],
    "Elastic Net selection rate (%)": [100 * enet_nonzero_features / X_endpoint.shape[1]],

    # Stability Selection
    "Stability runs": [N_STABILITY_RUNS],
    "Stability range 0.0": [(stability_score == 0.0).sum()],
    "Stability range 0.0-0.2": [((stability_score > 0.0) & (stability_score < 0.2)).sum()],
    "Stability range 0.2-0.4": [((stability_score >= 0.2) & (stability_score < 0.4)).sum()],
    "Stability range 0.4-0.6": [((stability_score >= 0.4) & (stability_score < 0.6)).sum()],
    "Stability range 0.6-0.8": [((stability_score >= 0.6) & (stability_score < 0.8)).sum()],
    "Stability range 0.8-1.0": [((stability_score >= 0.8) & (stability_score < 1.0)).sum()],
    "Stability range 1.0": [(stability_score == 1.0).sum()],
    "Features with stability > 0": [(stability_score > 0).sum()],
    f"Stable features (score >= {STABILITY_THRESHOLD})": [len(stable_features)],
    "Stability selection rate (%)": [100 * len(stable_features) / X_endpoint.shape[1]],

    # XGBoost
    "XGBoost N features": [X_xgb.shape[1]],
    "XGBoost CV metric mean": [xgb_cv_scores.mean()],
    "XGBoost CV metric SD": [xgb_cv_scores.std()],
    "XGBoost permutation observed": [xgb_perm_score],
    "XGBoost permutation p-value": [xgb_pvalue],
    "XGBoost signal": ["Yes" if XGB_HAS_SIGNAL else "No"],

    # SHAP & FSACI
    "SHAP used in FSACI": ["Yes" if USE_SHAP_IN_FSACI else "No"],
    "Model has signal": ["Yes" if MODEL_HAS_SIGNAL else "No"],
    "FSACI positive only": ["Yes" if PRIORITIZE_POSITIVE_ONLY else "No"],
    "Features with FSACI > 0": [(fsaci_rank["FSACI"] > 0).sum()],
    "FSACI max score": [fsaci_rank["FSACI"].max()],
    "FSACI top feature": [fsaci_rank.iloc[0]["ML_feature_id"]],
    "FSACI top score": [fsaci_rank.iloc[0]["FSACI"]],
    "FSACI top feature stability score": [fsaci_rank.iloc[0]["stability_score"]],

    # Run settings
    "CV folds x repeats": [f"{N_SPLITS} x {N_REPEATS}"],
    "N permutations": [N_PERM],
    "Fast mode": ["Yes" if FAST_MODE else "No"],
})

display(endpoint_stats.T)

minorsep()
logging.info(f"Saving endpoint statistics table | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

outfile_stat = (END_OUT_DIR / f"stats_{ENDPOINT_SHORT}.csv")
outfile_stat_T = (END_OUT_DIR / f"stats_{ENDPOINT_SHORT}_T.csv")
endpoint_stats.to_csv(outfile_stat, index=False)
endpoint_stats.T.to_csv(outfile_stat_T, index=True)

logging.info(f"Saved: {outfile_stat}")
minorsep()
logging.info(f"Saved: {outfile_stat_T}")

minorsep()
logging.info(f"Done building & saving endpoint statistics table | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# ML end


In [ ]:
# !!!!! WHEN ALL FSACI FILES ARE READY !!!!!
# Cross-endpoint statistics summary (cell rewritten)
# Run this cell first, then the "Cross-endpoint compound summary" cell (it uses the endpoint settings from here).
# - Reads stats_<endpoint>.csv from every endpoint folder
# - Decides which endpoints enter the cross-endpoint compound ranking: a model with signal AND a minimum
#   effect size (CV Q² >= MIN_Q2 or CV AUC >= MIN_AUC), so "significant but tiny" endpoints are left out
# - Endpoint weight = model quality on a 0-1 scale: Q² (regression) or (AUC - 0.5) / 0.5 (classification)
# - Endpoint blocks: endpoints measuring the same biology (e.g. the 4 IL endpoints) count once in the global score
# - Saves endpoint_model_summary.csv, endpoint_model_summary_T.csv and endpoint_model_summary_T.xlsx
#   (sheets: data, explanation)
majorsep()
logging.info(f"Building cross-endpoint statistics summary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter

MODELS_DIR = OUT_DIR / "endpoint_models"
SUMMARY_OUT_DIR = OUT_DIR / "cross_endpoint_summary"
SUMMARY_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Settings for the cross-endpoint compound ranking
MIN_Q2 = 0.10 # regression endpoints: best CV Q² of a model with signal must be >= this
MIN_AUC = 0.60 # classification endpoints: best CV AUC of a model with signal must be >= this

# Endpoints measuring the same biology form one block (counted once in the global score)
ENDPOINT_BLOCKS = {
    "cytotoxicity": ["cell_viability"],
    "inflammation": ["IL6_0.1", "IL6_1.0", "IL8_0.1", "IL8_1.0"],
    "AhR": ["AhR"],
    "androgen_receptor": ["andro", "antiandro"],
    "estrogen_receptor": ["estro", "antiestro"],
    "thyroid_receptor": ["TR"],
}

# Desired endpoint order (endpoint folders not listed here are added at the end)
endpoint_order = ["cell_viability", "IL6_0.1", "IL6_1.0", "IL8_0.1", "IL8_1.0",
    "andro", "antiandro", "estro", "antiestro", "AhR", "TR"]

if not MODELS_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {MODELS_DIR}")
extra_endpoints = sorted(d.name for d in MODELS_DIR.iterdir() if d.is_dir() and d.name not in endpoint_order)
endpoints_all = endpoint_order + extra_endpoints
endpoint_to_block = {e: b for b, eps in ENDPOINT_BLOCKS.items() for e in eps}


def write_xlsx_with_explanation(path, sheets, explanation):
    """Writes an .xlsx file. sheets = {sheet_name: (DataFrame, write_index)}; explanation = DataFrame
    with columns ["Item", "Explanation"], written as the last sheet 'explanation'."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, (df, write_index) in sheets.items():
            df.to_excel(writer, sheet_name=name, index=write_index)
            ws = writer.sheets[name]
            ws.freeze_panes = "B2" if write_index else "A2"
            for c in ws[1]:
                c.font = Font(bold=True)
            for i, col in enumerate(ws.iter_cols(min_row=1, max_row=min(ws.max_row, 200)), start=1):
                width = max((len(str(c.value)) for c in col if c.value is not None), default=8)
                ws.column_dimensions[get_column_letter(i)].width = min(max(10, width + 2), 45)
        explanation.to_excel(writer, sheet_name="explanation", index=False)
        ws = writer.sheets["explanation"]
        ws.column_dimensions["A"].width = 42
        ws.column_dimensions["B"].width = 120
        for row in ws.iter_rows(min_row=1):
            for c in row:
                c.alignment = Alignment(wrap_text=True, vertical="top")
        for c in ws[1]:
            c.font = Font(bold=True)


# Read statistics files
stats_tables = []
for endpoint in endpoints_all:
    file = MODELS_DIR / endpoint / f"stats_{endpoint}.csv"
    if not file.exists():
        logging.warning(f" - No statistics file found for {endpoint}")
        continue
    # tmp = pd.read_csv(file)
    tmp = safe_read_csv(file, "tmp")
    minorsep()
    tmp = tmp.loc[:, ~tmp.columns.str.contains("^Unnamed")]
    stats_tables.append(tmp)

if len(stats_tables) == 0:
    raise FileNotFoundError("No endpoint statistics files were found.")

stats_summary = pd.concat(stats_tables, ignore_index=True, sort=False)
stats_summary["Endpoint"] = stats_summary["Endpoint"].astype(str)
stats_summary["_order"] = stats_summary["Endpoint"].map({e: i for i, e in enumerate(endpoints_all)})
stats_summary = stats_summary.sort_values("_order").drop(columns="_order").reset_index(drop=True)

if stats_summary["Endpoint"].duplicated().any():
    raise ValueError(f"Duplicate endpoints in statistics files: {stats_summary['Endpoint'][stats_summary['Endpoint'].duplicated()].tolist()}")

# Make sure the columns used below exist
for col in ["Task", "Metric", "Chance level", "Total features", "Elastic Net training metric", "Elastic Net CV metric mean",
            "XGBoost CV metric mean", "Elastic Net permutation p-value", "XGBoost permutation p-value",
            "Elastic Net selected compounds (≠ 0)", "XGBoost N features", "Elastic Net signal", "XGBoost signal", "Model has signal"]:
    if col not in stats_summary.columns:
        stats_summary[col] = np.nan
num_cols_stats = ["Chance level", "Total features", "Elastic Net training metric", "Elastic Net CV metric mean",
                  "XGBoost CV metric mean", "Elastic Net permutation p-value", "XGBoost permutation p-value",
                  "Elastic Net selected compounds (≠ 0)", "XGBoost N features"]
stats_summary[num_cols_stats] = stats_summary[num_cols_stats].apply(pd.to_numeric, errors="coerce")

# Derived metrics
stats_summary["Best CV metric"] = stats_summary[["Elastic Net CV metric mean", "XGBoost CV metric mean"]].max(axis=1)
stats_summary["Best model"] = np.where(
    stats_summary["XGBoost CV metric mean"].fillna(-np.inf) > stats_summary["Elastic Net CV metric mean"].fillna(-np.inf),
    "XGBoost", "Elastic Net")
stats_summary["dMetric (XGBoost-Elastic Net)"] = stats_summary["XGBoost CV metric mean"] - stats_summary["Elastic Net CV metric mean"]
stats_summary["Elastic Net overfit (train - CV)"] = stats_summary["Elastic Net training metric"] - stats_summary["Elastic Net CV metric mean"]
stats_summary["Elastic Net selected (%)"] = 100 * stats_summary["Elastic Net selected compounds (≠ 0)"] / stats_summary["Total features"]
stats_summary["XGBoost input (%)"] = 100 * stats_summary["XGBoost N features"] / stats_summary["Total features"]
stats_summary["Permutation significant"] = np.where(
    (stats_summary["XGBoost permutation p-value"] < 0.05) | (stats_summary["Elastic Net permutation p-value"] < 0.05), "Yes", "No")

# Quality: Q² for regression, AUC for classification
is_clf = stats_summary["Task"].astype(str).eq("classification")
best = stats_summary["Best CV metric"]
stats_summary["Overall quality"] = np.select(
    [
        (~is_clf & (best >= 0.30)) | (is_clf & (best >= 0.75)),
        (~is_clf & (best >= 0.15)) | (is_clf & (best >= 0.65)),
        (~is_clf & (best > 0.00)) | (is_clf & (best > 0.55)),
    ],
    ["Good", "Moderate", "Weak"],
    default="Poor"
)

# Inclusion in the cross-endpoint compound ranking: model with signal AND minimum effect size
enet_ok = stats_summary["Elastic Net signal"].astype(str).eq("Yes")
xgb_ok = stats_summary["XGBoost signal"].astype(str).eq("Yes")
best_sig = pd.concat([stats_summary["Elastic Net CV metric mean"].where(enet_ok),
                      stats_summary["XGBoost CV metric mean"].where(xgb_ok)], axis=1).max(axis=1)
min_req = pd.Series(np.where(is_clf, MIN_AUC, MIN_Q2), index=stats_summary.index)
included = best_sig.fillna(-np.inf) >= min_req
weight_raw = np.where(is_clf, (best_sig - 0.5) / 0.5, best_sig)

stats_summary["Best CV metric (models with signal)"] = best_sig
stats_summary["Minimum CV metric required"] = min_req
stats_summary["Included in compound ranking"] = np.where(included, "Yes", "No")
reasons = []
for inc, b, m, metric in zip(included, best_sig, min_req, stats_summary["Metric"].astype(str)):
    if inc:
        reasons.append("")
    elif pd.isna(b):
        reasons.append("no model with signal")
    else:
        reasons.append(f"best CV {metric} {b:.3f} below minimum {m}")
stats_summary["Exclusion reason"] = reasons
stats_summary["Endpoint block"] = stats_summary["Endpoint"].map(endpoint_to_block).fillna(stats_summary["Endpoint"])
stats_summary["Endpoint weight"] = np.round(np.where(included, np.clip(weight_raw, 0, 1), 0.0), 3)

ENDPOINT_INFO = stats_summary.set_index("Endpoint") # used by the compound summary cell

key_cols = ["Endpoint", "Task", "Metric", "Endpoint count", "Elastic Net CV metric mean", "Elastic Net permutation p-value",
            "XGBoost CV metric mean", "XGBoost permutation p-value", "Best CV metric (models with signal)",
            "Overall quality", "Included in compound ranking", "Exclusion reason", "Endpoint block", "Endpoint weight"]
display(stats_summary[[c for c in key_cols if c in stats_summary.columns]])

minorsep()
logging.info(f" - Endpoints: {len(stats_summary)}")
logging.info(f" - Inclusion rule: model with signal and CV Q² >= {MIN_Q2} (regression) / CV AUC >= {MIN_AUC} (classification)")
for _, row in stats_summary.iterrows():
    if row["Included in compound ranking"] == "Yes":
        logging.info(f"   included: {row['Endpoint']} (block {row['Endpoint block']}, weight {row['Endpoint weight']:.3f})")
    else:
        logging.info(f"   excluded: {row['Endpoint']} ({row['Exclusion reason']})")

# Save csv
outfile = SUMMARY_OUT_DIR / "endpoint_model_summary.csv"
outfile_T = SUMMARY_OUT_DIR / "endpoint_model_summary_T.csv"
stats_summary.to_csv(outfile, index=False)
stats_summary.T.to_csv(outfile_T, header=False)

# Explanation sheet
STAT_EXPLANATIONS = {
    "Endpoint": "Short name of the tox endpoint (column header of the 'data' sheet).",
    "Task": "How the endpoint was modeled: 'regression' (predict the value) or 'classification' (predict active vs. inactive). Chosen automatically from the endpoint distribution.",
    "Target": "What the models predicted: log10(y) or log10(y + c) for skewed endpoints (c = half the smallest positive value, used when zeros occur), 'none' = raw values, 'binary: active = value above X' for classification (X = the lowest value, shared by many samples, e.g. the <LOQ substitute or 0 = no effect).",
    "Metric": "Performance metric: Q² for regression (R² of cross-validated predictions; 0 = no better than predicting the mean, 1 = perfect, negative = worse than the mean) or AUC for classification (0.5 = chance, 1 = perfect).",
    "Chance level": "Metric value expected without any signal (Q² = 0, AUC = 0.5).",
    "Endpoint count": "Number of samples with a value for this endpoint (used for modeling).",
    "N active (classification)": "Classification only: number of samples in the 'active' class.",
    "Endpoint mean": "Mean of the endpoint values (original scale, before any transformation).",
    "Endpoint median": "Median of the endpoint values (original scale).",
    "Endpoint SD": "Standard deviation of the endpoint values (original scale).",
    "Endpoint min": "Minimum endpoint value (original scale).",
    "Endpoint max": "Maximum endpoint value (original scale).",
    "N unique values": "Number of distinct endpoint values.",
    "% at most common value": "Share of samples at the single most common value. A high share usually means many values at the <LOQ substitute or at 'no effect', which limits regression.",
    "Total features": "Number of dust features used for modeling this endpoint (after the detection-frequency filter, minus features constant within this endpoint's samples).",
    "Elastic Net best alpha": "Elastic Net regularization strength chosen by cross-validation. Higher = fewer and smaller coefficients; the largest values mean (almost) no feature is useful.",
    "Elastic Net best l1_ratio": "Elastic Net mix chosen by cross-validation. Low (0.1) = ridge-like: small weights spread over many correlated features (diffuse signal). High (0.9) = lasso-like: few selected features (specific signal).",
    "Elastic Net training metric": "Metric on the same samples the model was fitted on. Always optimistic when there are far more features than samples; do not use it to judge the model.",
    "Elastic Net CV metric mean": "MAIN PERFORMANCE MEASURE: cross-validated metric (5 folds x 10 repeats; alpha re-tuned inside every training fold), mean over the repeats.",
    "Elastic Net CV metric SD": "Standard deviation of the CV metric between repeats (how stable the estimate is).",
    "Elastic Net permutation observed": "CV metric on the real data in the permutation test (one 5-fold split).",
    "Elastic Net permutation p-value": "Share of runs with shuffled endpoint values that scored at least as well as the real data. Smallest possible value = 1 / (N permutations + 1), e.g. 0.0099 with 100 permutations; report as p ≤ 0.01.",
    "Elastic Net signal": "Yes if the CV metric is above chance level and the permutation p-value < 0.05.",
    "Elastic Net selected compounds (≠ 0)": "Number of features with a non-zero Elastic Net coefficient in the final model.",
    "Elastic Net selected positive": "Of these, features with a positive coefficient: higher concentration goes with a higher endpoint value (more activity).",
    "Elastic Net selection rate (%)": "Selected features as % of all features.",
    "Stability runs": "Number of Elastic Net refits on random subsamples of the samples (stability selection).",
    "Features with stability > 0": "Features selected in at least one stability run.",
    "Stability selection rate (%)": "Stable features as % of all features.",
    "XGBoost N features": "Number of features given to XGBoost (the most stable features, max. 200). 0 = XGBoost not run because no feature was ever selected.",
    "XGBoost CV metric mean": "Cross-validated metric of XGBoost (5 folds x 10 repeats); feature selection is redone inside every training fold. Blank = XGBoost not run.",
    "XGBoost CV metric SD": "Standard deviation of the XGBoost CV metric between repeats.",
    "XGBoost permutation observed": "XGBoost CV metric on the real data in the permutation test.",
    "XGBoost permutation p-value": "XGBoost permutation p-value (see Elastic Net permutation p-value).",
    "XGBoost signal": "Yes if the XGBoost CV metric is above chance level and its permutation p-value < 0.05.",
    "SHAP used in FSACI": "Yes if XGBoost has signal: SHAP importance then contributes to FSACI. Otherwise FSACI uses the Elastic Net only.",
    "Model has signal": "Yes if Elastic Net or XGBoost has signal. Does not require a minimum effect size; see 'Included in compound ranking'.",
    "FSACI positive only": "Yes = FSACI only ranks features linked to higher endpoint values (more activity).",
    "Features with FSACI > 0": "Number of features prioritized for this endpoint.",
    "FSACI max score": "Highest FSACI value (0-1).",
    "FSACI top feature": "Feature with the highest FSACI. If 'Features with FSACI > 0' is 0, this is just the first feature in the table and has no meaning.",
    "FSACI top score": "FSACI of the top feature.",
    "FSACI top feature stability score": "Stability score of the top feature (share of stability runs selecting it).",
    "CV folds x repeats": "Cross-validation settings: folds x repeats.",
    "N permutations": "Number of permutations in the permutation tests.",
    "Fast mode": "Yes = quick test run with fewer repeats/permutations (less precise).",
    "Best CV metric": "Higher of the Elastic Net and XGBoost CV metric (regardless of significance).",
    "Best model": "Model with the higher CV metric.",
    "dMetric (XGBoost-Elastic Net)": "XGBoost minus Elastic Net CV metric. > 0 = XGBoost better (non-linear effects); < 0 = the linear model is sufficient.",
    "Elastic Net overfit (train - CV)": "Training minus CV metric. Large values = the model fits noise in the training samples (expected with many more features than samples).",
    "Elastic Net selected (%)": "Elastic Net selected features as % of all features.",
    "XGBoost input (%)": "XGBoost input features as % of all features.",
    "Permutation significant": "Yes if at least one model has a permutation p-value < 0.05.",
    "Overall quality": "Rough rating of 'Best CV metric'. Regression: Q² ≥ 0.30 Good, ≥ 0.15 Moderate, > 0 Weak, else Poor. Classification: AUC ≥ 0.75 Good, ≥ 0.65 Moderate, > 0.55 Weak, else Poor.",
    "Best CV metric (models with signal)": "Best CV metric among the models with signal. Blank = no model with signal.",
    "Minimum CV metric required": f"Minimum effect size for the cross-endpoint compound ranking: CV Q² ≥ {MIN_Q2} (regression) or CV AUC ≥ {MIN_AUC} (classification).",
    "Included in compound ranking": "Yes = endpoint used in the cross-endpoint compound ranking (compound_global_summary): a model with signal AND best CV metric ≥ the minimum.",
    "Exclusion reason": "Why the endpoint is not used in the cross-endpoint compound ranking.",
    "Endpoint block": "Group of endpoints measuring the same biology (e.g. all IL-6/IL-8 endpoints = inflammation). A block counts once in the global score.",
    "Endpoint weight": "Weight in the global score = model quality on a 0-1 scale: CV Q² (regression) or (CV AUC − 0.5) / 0.5 (classification) of the best model with signal; 0 if not included.",
}

def stat_explanation(name):
    if name in STAT_EXPLANATIONS:
        return STAT_EXPLANATIONS[name]
    if name.startswith("Stability range"):
        return "Number of features whose stability score (share of stability runs selecting the feature) falls in this range."
    if name.startswith("Stable features (score"):
        return "Number of features selected in at least this share of stability runs ('stable' features)."
    return "No description available."

blocks_txt = "; ".join(f"{b}: {', '.join(eps)}" for b, eps in ENDPOINT_BLOCKS.items())
explanation_stats = pd.DataFrame(
    [
        ("ABOUT THIS FILE", "One column per tox endpoint, one row per statistic of the endpoint models (Elastic Net, stability selection, XGBoost, FSACI) plus derived columns. Created by the cross-endpoint statistics summary cell of the toxicity-driven prioritization notebook."),
        ("HOW TO READ", "Judge a model by its cross-validated (CV) metric and the permutation p-value, not by the training metric. Q² (regression): 0 = no better than predicting the mean. AUC (classification): 0.5 = chance. An endpoint without a model with signal gives no reliable compound ranking; a model with signal but a very small CV metric is statistically real but of little practical use."),
        ("SETTINGS", f"Minimum CV Q² = {MIN_Q2}; minimum CV AUC = {MIN_AUC}. Endpoint blocks: {blocks_txt}."),
        ("", ""),
        ("ROWS OF THE 'data' SHEET", ""),
    ]
    + [(row_name, stat_explanation(row_name)) for row_name in stats_summary.columns],
    columns=["Item", "Explanation"]
)

stats_T_df = stats_summary.set_index("Endpoint").T
stats_T_df.index.name = "Variable"
outfile_xlsx = SUMMARY_OUT_DIR / "endpoint_model_summary_T.xlsx"
write_xlsx_with_explanation(outfile_xlsx, {"data": (stats_T_df, True)}, explanation_stats)

minorsep()
logging.info(f"Saved:\n - {outfile}\n - {outfile_T}\n - {outfile_xlsx}")

minorsep()
logging.info(f"Done building cross-endpoint statistics summary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Cross-endpoint compound summary (cell rewritten)
# Run AFTER the "Cross-endpoint statistics summary" cell (uses ENDPOINT_INFO, ENDPOINT_BLOCKS, write_xlsx_with_explanation).
# Fixes four biases of the old global score:
# - Weak endpoints left out: only endpoints with 'Included in compound ranking' = Yes (signal + minimum effect size)
# - Dense endpoints no longer dominate: each endpoint contributes at most its top MAX_RANKED_PER_ENDPOINT
#   compounds, scored by percentile rank within that set (1 = rank 1, close to 0 = last)
# - Strength over breadth: the global score is a weighted mean of block scores, so a compound must rank high,
#   not just appear somewhere in a long list
# - Same biology counted once: endpoints are grouped in blocks (e.g., 4 IL endpoints = 1 inflammation block)
#  Weights = model quality of each endpoint (from the previous cell)
# Saves compound_endpoint_summary(_w_metadata) and compound_global_summary(_w_metadata) as .csv,
# and both *_w_metadata files as .xlsx with 'endpoints' and 'explanation' sheets
majorsep()
logging.info(f"Building cross-endpoint compound summary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

if "ENDPOINT_INFO" not in globals() or "write_xlsx_with_explanation" not in globals():
    raise RuntimeError("Run the 'Cross-endpoint statistics summary' cell first")

MAX_RANKED_PER_ENDPOINT = 100 # each included endpoint contributes at most its top-100 prioritized compounds
TOP_N_LIST = [10, 25, 50] # counts of blocks where a compound is in the top N
KEY_META_COLS = ["feature_name", "class", "inchikey", "formula"] # metadata columns shown right after the ID


def summary_meta(df):
    """Adds feature metadata from the full classified metadata (feat_meta_class) or, if not loaded,
    feat_meta_dust_masked. Key metadata columns are placed right after ML_feature_id."""
    meta = None
    for meta_name in ["feat_meta_class", "feat_meta_dust_masked"]:
        m = globals().get(meta_name)
        if m is not None and "ML_feature_id" in m.columns:
            meta = m.copy()
            break
    if meta is None:
        logging.warning(" - No feature metadata loaded: files are saved without metadata")
        return df
    meta["ML_feature_id"] = meta["ML_feature_id"].astype(str).str.strip()
    meta = meta.drop_duplicates("ML_feature_id")
    meta = meta[["ML_feature_id"] + [c for c in meta.columns if c not in df.columns]]
    out = df.copy()
    out["ML_feature_id"] = out["ML_feature_id"].astype(str).str.strip()
    out = out.merge(meta, on="ML_feature_id", how="left")
    front = [c for c in out.columns if c in ["global_rank", "ML_feature_id"]] + [c for c in KEY_META_COLS if c in out.columns]
    out = out[front + [c for c in out.columns if c not in front]]
    n_missing = int(out["feature_name"].isna().sum()) if "feature_name" in out.columns else len(out)
    logging.info(f" - Metadata: {len(out) - n_missing:,} of {len(out):,} features matched")
    return out


# Read FSACI rankings of all endpoints
fsaci_ranks, n_prioritized = {}, {}
for endpoint in endpoints_all:
    file = MODELS_DIR / endpoint / f"FSACI_{endpoint}.csv"
    if not file.exists():
        logging.warning(f" - No FSACI file for {endpoint}")
        continue
    # tmp = pd.read_csv(file)
    tmp = safe_read_csv(file, "tmp")
    if not {"ML_feature_id", "FSACI"}.issubset(tmp.columns):
        logging.warning(f" - {file.name}: missing ML_feature_id/FSACI columns; skipped")
        continue
    tmp["ML_feature_id"] = tmp["ML_feature_id"].astype(str).str.strip()
    tmp["FSACI"] = pd.to_numeric(tmp["FSACI"], errors="coerce")
    pos = tmp[tmp["FSACI"] > 0].drop_duplicates("ML_feature_id").set_index("ML_feature_id")["FSACI"]
    fsaci_ranks[endpoint] = pos.rank(ascending=False, method="min")
    n_prioritized[endpoint] = len(pos)
minorsep()

if not fsaci_ranks or sum(n_prioritized.values()) == 0:
    raise FileNotFoundError(f"No prioritized compounds (FSACI > 0) found in {MODELS_DIR}")
    minorsep()

ep_cols = list(fsaci_ranks)
included_eps = [e for e in ep_cols
                if e in ENDPOINT_INFO.index and ENDPOINT_INFO.loc[e, "Included in compound ranking"] == "Yes"
                and n_prioritized[e] > 0]

logging.info(f"Endpoints with FSACI files: \n{ep_cols}\n")
logging.info(f"Endpoints used in the global ranking: \n{included_eps}")
minorsep()


# Compound endpoint summary: FSACI rank per endpoint (all endpoints), blank = not prioritized
endpoint_summary = pd.DataFrame(fsaci_ranks)
endpoint_summary.index.name = "ML_feature_id"
endpoint_summary["n_endpoints_prioritized"] = endpoint_summary[ep_cols].notna().sum(axis=1)
if included_eps:
    endpoint_summary["n_included_endpoints_prioritized"] = endpoint_summary[included_eps].notna().sum(axis=1)
    endpoint_summary["best_rank_included_endpoints"] = endpoint_summary[included_eps].min(axis=1)
else:
    endpoint_summary["n_included_endpoints_prioritized"] = 0
    endpoint_summary["best_rank_included_endpoints"] = np.nan
endpoint_summary[ep_cols] = endpoint_summary[ep_cols].astype("Int64")
endpoint_summary["best_rank_included_endpoints"] = endpoint_summary["best_rank_included_endpoints"].astype("Int64")
endpoint_summary = (
    endpoint_summary.reset_index()
    .sort_values(["best_rank_included_endpoints", "n_included_endpoints_prioritized", "n_endpoints_prioritized"],
                 ascending=[True, False, False], na_position="last")
    .reset_index(drop=True)
)
logging.info(f" - Compound endpoint summary: {len(endpoint_summary):,} features prioritized for >= 1 endpoint")


# Global prioritization (included endpoints only)
if not included_eps:
    raise RuntimeError("No endpoint passes the inclusion rules: lower MIN_Q2 / MIN_AUC in the previous cell")

ranks_inc = pd.DataFrame({e: fsaci_ranks[e] for e in included_eps})
scores = pd.DataFrame(index=ranks_inc.index)
n_used = {}
for e in included_eps:
    n_eff = min(n_prioritized[e], MAX_RANKED_PER_ENDPOINT)
    n_used[e] = n_eff
    r = ranks_inc[e]
    scores[e] = np.where(r <= n_eff, 1 - (r - 1) / n_eff, 0.0) # NaN rank -> 0

ep_weight = {e: float(ENDPOINT_INFO.loc[e, "Endpoint weight"]) for e in included_eps}
blocks_inc = {}
for e in included_eps:
    blocks_inc.setdefault(str(ENDPOINT_INFO.loc[e, "Endpoint block"]), []).append(e)

block_scores, block_weights, block_best_rank = {}, {}, {}
for b, eps in blocks_inc.items():
    w_b = max(ep_weight[e] for e in eps) # block weight = weight of the block's best-modeled endpoint
    block_weights[b] = w_b
    weighted = pd.DataFrame({e: scores[e] * ep_weight[e] for e in eps}) # endpoint score x endpoint weight
    block_scores[b] = weighted.max(axis=1) / w_b if w_b > 0 else weighted.max(axis=1) * 0
    block_best_rank[b] = ranks_inc[eps].where(scores[eps] > 0).min(axis=1) # best rank within the scored top set

total_weight = sum(block_weights.values())
if total_weight <= 0:
    raise RuntimeError("All endpoint weights are 0")
block_scores_df = pd.DataFrame(block_scores)
global_score = sum(block_weights[b] * block_scores_df[b] for b in block_scores) / total_weight

global_summary = pd.DataFrame({"ML_feature_id": scores.index})
global_summary["global_priority_score"] = global_score.values
global_summary["n_blocks_hit"] = (block_scores_df > 0).sum(axis=1).values
block_best_rank_df = pd.DataFrame(block_best_rank)
for N in TOP_N_LIST:
    global_summary[f"n_blocks_top{N}"] = (block_best_rank_df <= N).sum(axis=1).values
contrib = pd.DataFrame({b: block_weights[b] * block_scores_df[b] for b in block_scores})
global_summary["main_block"] = np.where(contrib.max(axis=1) > 0, contrib.idxmax(axis=1), "")
for b in block_scores:
    global_summary[f"block_score_{b}"] = block_scores_df[b].values
    global_summary[f"block_best_rank_{b}"] = block_best_rank_df[b].astype("Int64").values
for e in included_eps:
    global_summary[f"score_{e}"] = scores[e].values
    global_summary[f"rank_{e}"] = ranks_inc[e].astype("Int64").values

global_summary = global_summary[global_summary["global_priority_score"] > 0]
global_summary = (
    global_summary.sort_values(["global_priority_score", "n_blocks_hit"], ascending=False)
    .reset_index(drop=True)
)
global_summary.insert(0, "global_rank", np.arange(1, len(global_summary) + 1))
score_cols = [c for c in global_summary.columns if c.startswith(("global_priority", "block_score_", "score_"))]
global_summary[score_cols] = global_summary[score_cols].round(4)

# QC
logging.info(f"Global prioritization:")
logging.info(f" - Blocks used (weight): {', '.join(f'{b} ({block_weights[b]:.3f}: {blocks_inc[b]})' for b in block_weights)}")
logging.info(f" - Compounds per endpoint used in the score (max {MAX_RANKED_PER_ENDPOINT}): {n_used}")
logging.info(f" - Compounds in the global ranking: {len(global_summary):,}")
logging.info(f" - Compounds by number of blocks hit: {global_summary['n_blocks_hit'].value_counts().sort_index().to_dict()}")
top50_blocks = global_summary.head(50)["main_block"].value_counts().to_dict()
logging.info(f" - Main block of the top 50 compounds (no block should take all): {top50_blocks}")
minorsep()

global_summary_meta = summary_meta(global_summary)
endpoint_summary_meta = summary_meta(endpoint_summary)
minorsep()

show_cols = ["global_rank", "ML_feature_id"] + [c for c in KEY_META_COLS[:2] if c in global_summary_meta.columns] + \
            ["global_priority_score", "n_blocks_hit", "main_block"] + [f"rank_{e}" for e in included_eps]
logging.info(f"Top 20 compounds (global ranking):")
with pd.option_context("display.max_colwidth", 40):
    display(global_summary_meta[show_cols].head(20))
minorsep()


# Endpoints sheet (shared by both xlsx files)
endpoints_sheet = ENDPOINT_INFO.reset_index()[[
    "Endpoint", "Task", "Metric", "Best CV metric (models with signal)", "Minimum CV metric required",
    "Included in compound ranking", "Exclusion reason", "Endpoint block", "Endpoint weight"]].copy()
endpoints_sheet["N prioritized compounds (FSACI > 0)"] = endpoints_sheet["Endpoint"].map(n_prioritized).fillna(0).astype(int)
endpoints_sheet["N compounds used in global score"] = endpoints_sheet["Endpoint"].map(n_used).fillna(0).astype(int)
endpoints_sheet["Block weight"] = endpoints_sheet["Endpoint block"].map(block_weights)
endpoints_sheet.loc[endpoints_sheet["Included in compound ranking"] != "Yes", "Block weight"] = np.nan


# Explanation sheets
META_EXPLANATIONS = {
    "feature_name": "Compound / feature name from the classified feature metadata file.",
    "class": "Chemical class from the classified feature metadata file.",
    "inchikey": "InChIKey of the identified compound.",
    "formula": "Molecular formula.",
    "rt1_(min)": "Retention time 1 (min): first-dimension retention time (GCxGC-MS) or LC retention time.",
    "rt2_(sec)": "Retention time 2 (s): second-dimension retention time (GCxGC-MS only).",
    "feature_id": "Original feature ID from the lab.",
    "n_detected_samples": "Number of samples with a detection, as given in the metadata file (may differ slightly from the counts used in this pipeline).",
    "detection_frequency": "Detection frequency, as given in the metadata file.",
    "concentration_type": "Type of quantitative value (e.g. estimated concentration, normalized peak area).",
}
FSACI_TXT = ("FSACI (Final Stability-Adjusted Compound Index) = stability score x effect score. Stability score = share of "
             "Elastic Net stability runs selecting the compound; effect score = percentile rank of its Elastic Net "
             "coefficient and, if XGBoost had signal, its SHAP importance. Only compounds linked to HIGHER endpoint "
             "values (more activity) get FSACI > 0.")
CAUTION_TXT = ("Rankings show statistical associations, not proof that a compound causes the effect: a compound can rank high "
               "because it co-occurs with the active compounds (same sources). Unidentified features can rank high too. "
               "Confirm top candidates, e.g. with literature, effect-directed analysis or tests of the pure compounds.")
blocks_used_txt = "; ".join(f"{b}: {', '.join(blocks_inc[b])} (block weight {block_weights[b]:.3f})" for b in blocks_inc)
excluded_txt = "; ".join(f"{r['Endpoint']}: {r['Exclusion reason']}"
                         for _, r in endpoints_sheet.iterrows() if r["Included in compound ranking"] != "Yes") or "none"

def column_explanation(col):
    if col in META_EXPLANATIONS:
        return META_EXPLANATIONS[col]
    fixed = {
        "ML_feature_id": "Feature ID as used in the dust data and the feature metadata.",
        "global_rank": "Position in the global ranking (1 = highest global priority score).",
        "global_priority_score": "Global priority score (0-1): weighted mean of the block scores, weights = block weights (model quality). 1 = rank 1 in every block; a compound that is rank 1 in only one block gets that block's weight / sum of block weights.",
        "n_blocks_hit": f"Number of blocks in which the compound is among the top {MAX_RANKED_PER_ENDPOINT} prioritized compounds of at least one endpoint.",
        "main_block": "Block contributing most to the global score.",
        "n_endpoints_prioritized": "Number of endpoints (all endpoints) with FSACI > 0 for this compound.",
        "n_included_endpoints_prioritized": "Number of endpoints used in the global ranking with FSACI > 0 for this compound.",
        "best_rank_included_endpoints": "Best (lowest) FSACI rank among the endpoints used in the global ranking. Blank = prioritized only for endpoints not used in the global ranking.",
    }
    if col in fixed:
        return fixed[col]
    if col.startswith("n_blocks_top"):
        return f"Number of blocks in which the compound is in the top {col.replace('n_blocks_top', '')} of at least one endpoint."
    if col.startswith("block_score_"):
        b = col.replace("block_score_", "")
        return (f"Score for block '{b}' (0-1): highest endpoint score in the block, each endpoint score multiplied by its endpoint "
                f"weight and divided by the block weight (so a better-modeled endpoint counts more). 0 = not in the top set of any endpoint in the block.")
    if col.startswith("block_best_rank_"):
        return f"Best FSACI rank in block '{col.replace('block_best_rank_', '')}' (within the top {MAX_RANKED_PER_ENDPOINT} of each endpoint). Blank = not in the top set."
    if col.startswith("score_"):
        e = col.replace("score_", "")
        return f"Endpoint score for '{e}' (0-1) = 1 − (rank − 1) / {n_used.get(e, '?')}, for its top {n_used.get(e, '?')} compounds; 0 = not in this set."
    if col.startswith("rank_"):
        e = col.replace("rank_", "")
        return f"FSACI rank for '{e}' among its {n_prioritized.get(e, '?')} prioritized compounds (1 = highest). Blank = not prioritized."
    if col in ep_cols:
        inc = "yes" if col in included_eps else "no (see 'endpoints' sheet)"
        return f"FSACI rank for endpoint '{col}' among its {n_prioritized[col]} prioritized compounds (1 = highest priority). Blank = not prioritized. Used in the global ranking: {inc}."
    return "Feature metadata (from the feature metadata file)."

ENDPOINTS_SHEET_TXT = ("One row per endpoint: model quality, whether it is used in the global ranking (and why not), its block "
                       "and weight, and how many compounds it prioritizes / contributes to the global score.")

explanation_global = pd.DataFrame(
    [
        ("ABOUT THIS FILE", f"Cross-endpoint prioritization of dust features. One row per feature that is among the top {MAX_RANKED_PER_ENDPOINT} prioritized compounds (FSACI > 0) of at least one endpoint used in the global ranking. Sorted by global priority score."),
        ("ENDPOINTS USED", f"Endpoints with a model with signal (CV metric above chance, permutation p < 0.05) AND CV Q² ≥ {MIN_Q2} (regression) or CV AUC ≥ {MIN_AUC} (classification). Blocks: {blocks_used_txt}."),
        ("ENDPOINTS NOT USED", excluded_txt),
        ("HOW THE SCORE IS COMPUTED", f"1) Per endpoint: its top n compounds by FSACI (n = number prioritized, max {MAX_RANKED_PER_ENDPOINT}) get an endpoint score = 1 − (rank − 1) / n, i.e. 1 for rank 1 down to close to 0 for the last; all others 0. Each endpoint contributes the same maximum number of compounds, so endpoints with very long lists do not dominate. 2) Endpoints measuring the same biology form a block that counts once; block score = best weighted endpoint score in the block. 3) Global priority score = weighted mean of the block scores; weight = model quality."),
        ("ENDPOINT WEIGHT", "Model quality on a 0-1 scale: CV Q² (regression) or (CV AUC − 0.5) / 0.5 (classification) of the best model with signal. Block weight = weight of the block's best-modeled endpoint."),
        ("HOW TO READ", "A high score means the compound ranks high for well-modeled endpoints. Check n_blocks_hit: 1 block = endpoint-specific candidate; several blocks = candidate active in several assays. Per-endpoint ranks are in the rank_ columns and in compound_endpoint_summary."),
        ("FSACI", FSACI_TXT),
        ("CAUTION", CAUTION_TXT),
        ("SHEET 'endpoints'", ENDPOINTS_SHEET_TXT),
        ("", ""),
        ("COLUMNS OF THE 'data' SHEET", ""),
    ]
    + [(c, column_explanation(c)) for c in global_summary_meta.columns],
    columns=["Item", "Explanation"]
)

explanation_endpoint = pd.DataFrame(
    [
        ("ABOUT THIS FILE", "FSACI ranks of all dust features prioritized (FSACI > 0) for at least one tox endpoint, for every modeled endpoint, including endpoints not used in the global ranking."),
        ("HOW TO READ", "Rank 1 = highest priority within that endpoint. Ranks are only comparable within one endpoint: endpoints prioritize very different numbers of compounds (see 'endpoints' sheet), e.g. rank 50 of 95 vs. rank 50 of 2,000. Blank = not prioritized for that endpoint (not selected by the models, or linked to lower activity). Ranks for endpoints not used in the global ranking are not reliable."),
        ("SORTING", "By best rank among the endpoints used in the global ranking, then by the number of those endpoints prioritizing the compound."),
        ("FSACI", FSACI_TXT),
        ("CAUTION", CAUTION_TXT),
        ("SHEET 'endpoints'", ENDPOINTS_SHEET_TXT),
        ("", ""),
        ("COLUMNS OF THE 'data' SHEET", ""),
    ]
    + [(c, column_explanation(c)) for c in endpoint_summary_meta.columns],
    columns=["Item", "Explanation"]
)


# Save
endpoint_summary.to_csv(SUMMARY_OUT_DIR / "compound_endpoint_summary.csv", index=False)
endpoint_summary_meta.to_csv(SUMMARY_OUT_DIR / "compound_endpoint_summary_w_metadata.csv", index=False)
global_summary.to_csv(SUMMARY_OUT_DIR / "compound_global_summary.csv", index=False)
global_summary_meta.to_csv(SUMMARY_OUT_DIR / "compound_global_summary_w_metadata.csv", index=False)

write_xlsx_with_explanation(SUMMARY_OUT_DIR / "compound_endpoint_summary_w_metadata.xlsx",
                            {"data": (endpoint_summary_meta, False), "endpoints": (endpoints_sheet, False)},
                            explanation_endpoint)
write_xlsx_with_explanation(SUMMARY_OUT_DIR / "compound_global_summary_w_metadata.xlsx",
                            {"data": (global_summary_meta, False), "endpoints": (endpoints_sheet, False)},
                            explanation_global)

logging.info(f"Saved to {SUMMARY_OUT_DIR}:")
for f in ["compound_endpoint_summary.csv", "compound_endpoint_summary_w_metadata.csv/.xlsx",
          "compound_global_summary.csv", "compound_global_summary_w_metadata.csv/.xlsx"]:
    logging.info(f" - {f}")

minorsep()
logging.info(f"Done building cross-endpoint compound summary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()
